# HITL-IoT v1.1 — Step 8
## External Validation with CICIoT2023 and ToN-IoT Network

**Canonical notebook:** `05_HITL-IoT_v1.1_External_Validation.ipynb`

### Purpose

Reviewer 3 requested an **independent source of evidence** showing how HITL-IoT relates to established capture/testbed IoT security datasets. This notebook performs that evaluation without changing the frozen HITL-IoT v1.1 generator, labels, oversight subset, or Step-6/Step-7 results.

The external-validation design has four layers:

1. **Source and schema audit** — verify the supplied CICIoT2023 and ToN-IoT files and document label prevalence.
2. **Distributional correspondence and domain gap** — compare semantically harmonized features using class-conditional KS statistics, Jensen–Shannon divergence, normalized Wasserstein distance, correlation structure, PCA, and domain-discriminator AUROC.
3. **Cross-domain transfer** — train on one domain and test on another using the same small, predeclared harmonized feature set, with no target-label tuning.
4. **External calibration/selective prediction** — test whether probability calibration and uncertainty-based selective prediction remain useful on the independent external datasets.

### Interpretation boundary

This notebook does **not** try to prove that synthetic HITL-IoT traffic is identical to capture-based traffic. Domain differences are expected and are scientifically informative. The goal is to quantify correspondence, domain shift, transferability, and the portability of AI-side calibration/deferral mechanisms.

The ToN-IoT **Network** file is the primary ToN-IoT comparator. The uploaded Fridge, Garage Door, GPS Tracker, Modbus, Motion Light, Thermostat, and Weather files are telemetry datasets with different feature semantics; they are inventoried but are **not forced into a flow-feature transfer experiment**.

### External sources

- CICIoT2023 official page: https://www.unb.ca/cic/datasets/iotdataset-2023.html
- CICIoT2023 paper: Neto et al., *Sensors* 2023, 23(13):5941.
- ToN-IoT official page: https://research.unsw.edu.au/projects/toniot-datasets
- ToN-IoT Network: Moustafa, *Sustainable Cities and Society* 2021, 102994.
- ToN-IoT heterogeneity/standardization: Booij et al., *IEEE Internet of Things Journal* 2021.

CICIoT2023 reports 33 attacks across 105 IoT devices. ToN-IoT was collected from a realistic Industry 4.0 IoT/IIoT testbed spanning IoT, Cloud, and Edge/Fog components. These external resources therefore provide a deliberately broader comparison than HITL-IoT's controlled five-device smart-home scope.

### Execution requirement

For the final run use **Kernel → Restart & Run All**. Do not resume from a
later cell after an earlier exception, because later analyses depend on objects
created sequentially during provenance, inventory, harmonization, split
construction, duplicate audit, transfer, calibration, and figure generation.


## Pre-specified analysis principles

- **Primary HITL split:** frozen Step-6 temporal train/validation/test.
- **External internal splits:** fixed-seed stratified 70/15/15 because the supplied external CSVs do not expose a directly comparable chronological benchmark split.
- **No target-label tuning:** cross-domain model parameters and decision thresholds are selected only on the source-domain training/validation data.
- **Binary task only:** benign/attack is used for cross-domain transfer because attack taxonomies differ substantially.
- **Feature mapping must be semantic, not name-based.**
- **Main transfer feature set:** a conservative universal core available in all three domains.
- **Pairwise enriched feature sets:** reported as supplementary sensitivity analyses.
- **Primary models:** Logistic Regression and Random Forest. They are intentionally fixed rather than extensively tuned on each external dataset.
- **Native class prevalence is retained for final test metrics.**
- **Heavy-tailed magnitude features are transformed using `log1p` inside a source-fitted pipeline.**
- **Calibration/selective prediction is evaluated within each domain and under cross-domain shift.**

**ToN temporal note.** Chronological sensitivity is supplementary and conditional on the supplied ToN-IoT network CSV containing a usable timestamp. If no timestamp exists, the notebook records the analysis as unavailable and retains the fixed-seed stratified external split.

## Revision notes — Q1 final external-validation protocol

This version incorporates the final methodological safeguards requested during the Step-8 review:

1. **CICIoT2023 naming and provenance.** The external `IoT_Intrusion.csv` file is identified throughout as **CICIoT2023**, consistent with the source page that attributes the dataset to the CICIoT2023 study. Exact local filename and SHA-256 are retained in the reproducibility audit.
2. **Strict duplicate-leakage audit.** Exact harmonized feature vectors are counted per domain. In-domain test results are additionally reported on a **development-novel test subset** whose feature vectors occur in neither the training nor validation partition.
3. **Duplicate-robust calibration and selective prediction.** Calibration quality, exact-budget selective prediction, and AURC are reported both on the full test partition and on development-novel test records.
4. **Duplicate-robust domain discrimination.** Domain-discriminator experiments first collapse exact harmonized vectors within each domain, preventing a nonlinear discriminator from benefiting from repeated-vector train/test memorization.
5. **TTL excluded from all models.** HITL-IoT `ttl` is a per-device constant by design, so it is not used in the universal or enriched transfer feature sets.
6. **Destination port represented categorically.** The HITL↔ToN enriched sensitivity uses broad destination-port bands (well-known / registered / dynamic) rather than treating port numbers as a continuous magnitude.
7. **Compute and disk bounded.** Random Forests use `min_samples_leaf=20`. The three universal-feature within-domain Random-Forest reference models are always serialized; all additional transfer models are saved only when `SAVE_ALL_TRANSFER_MODELS=True`.
8. **Undefined MCC remains undefined.** MCC is reported as `NaN` when a test subset or prediction contains only one class. Validation-threshold selection converts undefined tie-break metrics to `-inf` internally rather than silently treating them as zero.
9. **Feature-semantics caveats are explicit.** CIC packet-window aggregates are treated as semantically harmonized approximations, not one-to-one flow-feature equivalents.
10. **ToN-IoT chronological sensitivity is conditional.** It runs only if the supplied ToN network file contains a usable timestamp field. The uploaded `train_test_network.csv` used for this revision does not expose `ts`, so the notebook records the temporal sensitivity as *not available* rather than implying it was completed.
11. **Publication figures** retain the Step-6/Step-7 journal style system and now distinguish headline full-test results from duplicate-robust development-novel sensitivity results.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, spearmanr, wasserstein_distance

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, log_loss,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler, StandardScaler
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXPECTED_HITL_SHA256 = "c6dd623029f6ca0967773379b2afbba96b73ab9cb2028c5e81ef89a1b15e1291"
LEGACY_EXPECTED_STEP6_FINGERPRINT = "69b553133c156aa9c37a45d69d0a75ec9dc2eafb29a128294df9ec35b5b299f5"

# Scientific invariants frozen for the canonical Step-6 benchmark.
# These are stricter and more meaningful than one composite fingerprint,
# which may change if notebook/protocol metadata changes.
EXPECTED_STEP6_INVARIANTS = {
    "dataset_sha256": "c6dd623029f6ca0967773379b2afbba96b73ab9cb2028c5e81ef89a1b15e1291",
    "feature_policy_sha256": "0666306f134fabe252bb87ae61d5fa1819f50203ef4382a18b8a6fa25306282f",
    "splits_sha256": "6a0c19d3c957b92f3eefa53267eeca140f490cd2031637bcda9be8908c8166b8",
    "schema_sha256": "7967ffa9a8ac51d6bf81788c3551a606e3f73dc255f41c9320ec3585fb3285e5",
    "seed": 42,
    "threshold": 0.5,
    "feature_policy_key": "track1_primary_flow_only",
}
EXPECTED_STEP6_REFERENCE_MODEL = "Random Forest"
EXPECTED_STEP6_MODEL_COUNT = 9

# Runtime controls.
FAST_MODE = False
FAST_CIC_TRAIN_CAP = 300_000
DIST_SAMPLE_PER_CLASS = 25_000
PCA_SAMPLE_PER_CLASS = 4_000
DOMAIN_CLASSIFIER_PER_CLASS = 10_000
RF_TREES = 300
RF_MIN_SAMPLES_LEAF = 20            # bounds tree size on ~734k CIC training rows (memory/disk), same for every domain
SAVE_ALL_TRANSFER_MODELS = False    # True also saves every transfer-matrix model (large); reference models always saved
RUN_TON_TEMPORAL_SENSITIVITY = True # runs only if a usable ToN timestamp field is present
SELECTIVE_BUDGETS = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30]
ECE_BINS = 15

OUT_DIR = Path("step8_external_validation_v1.1")
for d in ["audit", "tables", "figures", "models", "probabilities", "manuscript_values", "logs"]:
    (OUT_DIR / d).mkdir(parents=True, exist_ok=True)

def log1p_clip_array(x):
    """Convenience helper for exploratory use; persisted pipelines use np.log1p directly."""
    return np.log1p(np.clip(np.asarray(x, dtype=float), 0, None))

print("Step 8 configuration loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# Publication figure system — identical conventions to Figures 7–12 (Steps 6–7):
# journal sans-serif, 7.5–10 pt text, minimal spines, subtle grids, fixed
# colour-blind-safe semantics, vector PDF/SVG + 600-dpi PNG, (a)–(d) labels.
# ---------------------------------------------------------------------------
import re
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, PercentFormatter

SINGLE_COL, DOUBLE_COL = 3.5, 7.16
DOMAIN_ORDER = ["HITL-IoT", "CICIoT2023", "ToN-IoT"]
DOMAIN_COLORS = {"HITL-IoT": "#0072B2", "CICIoT2023": "#D55E00", "ToN-IoT": "#009E73"}
DOMAIN_MARKERS = {"HITL-IoT": "o", "CICIoT2023": "s", "ToN-IoT": "^"}
MODEL_COLORS = {"Logistic Regression": "#56B4E9", "Random Forest": "#1F1F1F"}
BENIGN_C, ATTACK_C = "#0072B2", "#D55E00"
NEUTRAL, GRID = "#4D4D4D", "#E3E3E3"
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", ["#F4F7FA", "#B9CEE0", "#6A96BD", "#2B5C8A", "#15324F"])
DIV_CMAP = LinearSegmentedColormap.from_list("div", ["#B3441B", "#E7A77E", "#F7F7F7", "#8FB3D0", "#2B5C8A"])

def set_publication_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "legend.fontsize": 7.5, "legend.frameon": False, "legend.handlelength": 2.0,
        "axes.linewidth": 0.6, "axes.edgecolor": "#333333", "axes.spines.top": False,
        "axes.spines.right": False, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 1.3,
        "figure.dpi": 110, "savefig.dpi": 600, "pdf.fonttype": 42, "ps.fonttype": 42,
        "svg.fonttype": "none", "axes.formatter.use_mathtext": False})

def ygrid(ax, axis="y"):
    ax.grid(True, axis=axis, color=GRID, linewidth=0.5); ax.set_axisbelow(True)

def add_panel_label(ax, letter, x=-0.16, y=1.04):
    ax.text(x, y, f"({letter})", transform=ax.transAxes, fontsize=10.5, fontweight="bold", va="bottom", ha="left")

FIGURE_REGISTRY = {}

def save_publication_figure(fig, stem, caption, n_panels):
    paths = {}
    for ext, kw in (("pdf", {}), ("svg", {}), ("png", {"dpi": 600})):
        p = OUT_DIR / "figures" / f"{stem}.{ext}"
        fig.savefig(p, bbox_inches="tight", pad_inches=0.03, **kw)
        paths[ext] = p
    fig.canvas.draw()
    FIGURE_REGISTRY[stem] = {
        "paths": paths, "caption": caption, "n_panels": n_panels,
        "size_in": tuple(np.round(fig.get_size_inches(), 2)),
        "axes": [{"xlabel": a.get_xlabel(), "ylabel": a.get_ylabel(),
                  "named_ticks": any(re.search(r"[A-Za-z]", t.get_text() or "")
                                     for t in a.get_xticklabels() + a.get_yticklabels())}
                 for a in fig.axes if a.axison and a.get_label() != "<colorbar>"],
        "texts": " ".join(t.get_text() for a in fig.axes for t in [a.xaxis.label, a.yaxis.label, *a.texts])}
    return paths

set_publication_style()
print("Publication figure style loaded:", mpl.font_manager.FontProperties(family=mpl.rcParams["font.sans-serif"]).get_name())


## File discovery

The notebook recognizes both the original filenames and the `(1)` filenames created when duplicate uploads are downloaded.  
The user's screenshot shows the frozen HITL-IoT release at `C:\Users\pc\release_v1.1`; that path is included as a candidate but is never trusted without SHA-256 verification.

### Step-6 lock policy

Step 8 validates the frozen Step-6 benchmark by its scientific invariants rather than by one brittle composite context fingerprint. The accepted package must match the frozen dataset, feature-policy, split, and schema SHA-256 values; seed; decision threshold; `track1_primary_flow_only` policy; all-nine validation/test probability handoff; and validation-selected Random-Forest reference model.

The observed composite context fingerprint is still recorded for auditability.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

def find_hitl_release(candidates):
    checked = []
    for root in candidates:
        root = Path(root)
        p = root / "HITL-IoT_dataset_v1.1.csv"
        if p.exists():
            observed = sha256_file(p)
            checked.append((str(root), observed))
            if observed == EXPECTED_HITL_SHA256:
                return root, checked
    return None, checked

def validate_step6_candidate(root):
    """
    Validate Step 6 by frozen scientific invariants and probability handoff.
    Composite fingerprint is recorded but is not the sole acceptance criterion.
    """
    root = Path(root)
    report = {
        "path": str(root),
        "exists": root.exists(),
        "context_fingerprint": None,
        "valid": False,
        "problems": [],
    }
    if not root.exists():
        report["problems"].append("directory missing")
        return False, report

    ctx_path = root / "logs" / "benchmark_context.json"
    ref_path = root / "manuscript_values" / "reference_model_selection.json"
    pval_path = root / "probabilities" / "temporal_validation_probabilities_all9.csv"
    ptest_path = root / "probabilities" / "temporal_test_probabilities_all9.csv"

    required = [ctx_path, ref_path, pval_path, ptest_path]
    missing = [str(p.relative_to(root)) for p in required if not p.exists()]
    if missing:
        report["problems"].append("missing: " + ", ".join(missing))
        return False, report

    # Context/invariant verification.
    try:
        ctx = json.loads(ctx_path.read_text(encoding="utf-8"))
    except Exception as exc:
        report["problems"].append(f"context parse error: {exc}")
        return False, report

    report["context_fingerprint"] = ctx.get("context_fingerprint")
    report["context"] = {
        k: ctx.get(k) for k in EXPECTED_STEP6_INVARIANTS
    }

    invariant_mismatches = {}
    for key, expected in EXPECTED_STEP6_INVARIANTS.items():
        observed = ctx.get(key)
        if key == "threshold" and observed is not None:
            try:
                observed = float(observed)
            except Exception:
                pass
        if observed != expected:
            invariant_mismatches[key] = {
                "expected": expected,
                "observed": observed,
            }

    if invariant_mismatches:
        report["problems"].append(
            "scientific invariant mismatch: "
            + json.dumps(invariant_mismatches, sort_keys=True)
        )

    # Reference-model verification.
    try:
        ref = json.loads(ref_path.read_text(encoding="utf-8"))
        report["reference_model"] = ref.get("selected_model")
        if ref.get("selected_model") != EXPECTED_STEP6_REFERENCE_MODEL:
            report["problems"].append(
                f"reference model {ref.get('selected_model')!r} != "
                f"{EXPECTED_STEP6_REFERENCE_MODEL!r}"
            )
    except Exception as exc:
        report["problems"].append(f"reference-model parse error: {exc}")

    # All-nine probability handoff verification.
    try:
        pv = pd.read_csv(
            pval_path,
            usecols=["flow_id", "model", "y_probability_attack", "y_true"]
        )
        pt = pd.read_csv(
            ptest_path,
            usecols=["flow_id", "model", "y_probability_attack", "y_true"]
        )
        probability_checks = {
            "validation_model_count": int(pv["model"].nunique()),
            "test_model_count": int(pt["model"].nunique()),
            "validation_duplicate_pairs":
                int(pv.duplicated(["flow_id", "model"]).sum()),
            "test_duplicate_pairs":
                int(pt.duplicated(["flow_id", "model"]).sum()),
            "validation_probability_range_ok":
                bool(pv["y_probability_attack"].between(0, 1).all()),
            "test_probability_range_ok":
                bool(pt["y_probability_attack"].between(0, 1).all()),
        }
        report["probability_checks"] = probability_checks

        if probability_checks["validation_model_count"] != EXPECTED_STEP6_MODEL_COUNT:
            report["problems"].append(
                f"validation has {probability_checks['validation_model_count']} models"
            )
        if probability_checks["test_model_count"] != EXPECTED_STEP6_MODEL_COUNT:
            report["problems"].append(
                f"test has {probability_checks['test_model_count']} models"
            )
        if probability_checks["validation_duplicate_pairs"] != 0:
            report["problems"].append("validation probability file has duplicate (flow_id, model) pairs")
        if probability_checks["test_duplicate_pairs"] != 0:
            report["problems"].append("test probability file has duplicate (flow_id, model) pairs")
        if not probability_checks["validation_probability_range_ok"]:
            report["problems"].append("validation probabilities outside [0,1]")
        if not probability_checks["test_probability_range_ok"]:
            report["problems"].append("test probabilities outside [0,1]")
    except Exception as exc:
        report["problems"].append(f"probability handoff error: {exc}")

    report["valid"] = len(report["problems"]) == 0
    return report["valid"], report

def find_step6(candidates):
    checked = []
    for root in candidates:
        ok, report = validate_step6_candidate(root)
        checked.append(report)
        if ok:
            return Path(root), checked
    return None, checked

# Manual overrides if your local folders are elsewhere.
HITL_RELEASE_OVERRIDE = None
STEP6_DIR_OVERRIDE = None
STEP7_DIR_OVERRIDE = None
CIC_FILE_OVERRIDE = None
TON_NETWORK_OVERRIDE = None
TON_TELEMETRY_DIR_OVERRIDE = None

hitl_candidates = [
    Path("data/v1.1/dataset"),
    Path("../data/v1.1/dataset"),
    Path("data/v1.1"),
    Path(r"C:\Users\pc\release_v1.1"),
    Path("release_v1.1"),
    Path("data") / "release_v1.1",
]
if HITL_RELEASE_OVERRIDE:
    hitl_candidates.insert(0, Path(HITL_RELEASE_OVERRIDE))
HITL_RELEASE, HITL_CHECKS = find_hitl_release(hitl_candidates)

# Prefer the frozen review-revision branch, but accept any candidate only if it
# passes the exact same scientific-invariant checks.
step6_candidates = [
    Path("track1_benchmark_v1.1_review_revision"),
    Path("track1_benchmark_v1.1_final"),
]
if STEP6_DIR_OVERRIDE:
    step6_candidates.insert(0, Path(STEP6_DIR_OVERRIDE))
STEP6_DIR, STEP6_CHECKS = find_step6(step6_candidates)

STEP7_DIR = first_existing([
    Path(STEP7_DIR_OVERRIDE) if STEP7_DIR_OVERRIDE else Path("__missing__"),
    Path("step7_selective_deferral_calibration_oversight_v1.1"),
])

CIC_FILE = first_existing([
    Path(CIC_FILE_OVERRIDE) if CIC_FILE_OVERRIDE else Path("__missing__"),
    Path("IoT_Intrusion.csv"),
    Path("IoT_Intrusion(1).csv"),
])

TON_NETWORK_FILE = first_existing([
    Path(TON_NETWORK_OVERRIDE) if TON_NETWORK_OVERRIDE else Path("__missing__"),
    Path("train_test_network.csv"),
    Path("train_test_network(1).csv"),
])

TELEMETRY_NAMES = {
    "Fridge": ["Train_Test_IoT_Fridge.csv", "Train_Test_IoT_Fridge(1).csv"],
    "Garage Door": ["Train_Test_IoT_Garage_Door.csv", "Train_Test_IoT_Garage_Door(1).csv"],
    "GPS Tracker": ["Train_Test_IoT_GPS_Tracker.csv", "Train_Test_IoT_GPS_Tracker(1).csv"],
    "Modbus": ["Train_Test_IoT_Modbus.csv", "Train_Test_IoT_Modbus(1).csv"],
    "Motion Light": ["Train_Test_IoT_Motion_Light.csv", "Train_Test_IoT_Motion_Light(1).csv"],
    "Thermostat": ["Train_Test_IoT_Thermostat.csv", "Train_Test_IoT_Thermostat(1).csv"],
    "Weather": ["Train_Test_IoT_Weather.csv", "Train_Test_IoT_Weather(1).csv"],
}

telemetry_files = {}
telemetry_root = (
    Path(TON_TELEMETRY_DIR_OVERRIDE)
    if TON_TELEMETRY_DIR_OVERRIDE else Path(".")
)
for name, candidates in TELEMETRY_NAMES.items():
    p = first_existing([telemetry_root / c for c in candidates])
    if p is not None:
        telemetry_files[name] = p

assert HITL_RELEASE is not None, (
    "Frozen canonical HITL-IoT v1.1 release not found. "
    f"Expected dataset SHA-256={EXPECTED_HITL_SHA256}. Checked={HITL_CHECKS}"
)

assert STEP6_DIR is not None, (
    "No Step-6 directory passed the frozen scientific-invariant lock.\n"
    + json.dumps(STEP6_CHECKS, indent=2, default=str)
)

assert CIC_FILE is not None, "CICIoT2023 IoT_Intrusion.csv not found."
assert TON_NETWORK_FILE is not None, "ToN-IoT train_test_network.csv not found."

STEP6_SELECTED_REPORT = next(
    r for r in STEP6_CHECKS if r.get("path") == str(STEP6_DIR)
)

print("[PASS] HITL release:", HITL_RELEASE)
print("[PASS] Step 6:", STEP6_DIR)
print("[PASS] Step-6 scientific invariants verified.")
print("[INFO] Observed Step-6 context fingerprint:",
      STEP6_SELECTED_REPORT.get("context_fingerprint"))
if STEP6_SELECTED_REPORT.get("context_fingerprint") != LEGACY_EXPECTED_STEP6_FINGERPRINT:
    print(
        "[INFO] Composite fingerprint differs from the earlier freeze note, "
        "but the dataset/feature-policy/split/schema hashes, seed, threshold, "
        "Track-1 policy, all-nine probabilities, and RF reference selection match."
    )
print("[PASS] CICIoT2023:", CIC_FILE)
print("[PASS] ToN-IoT Network:", TON_NETWORK_FILE)
print("[INFO] ToN telemetry files found:", list(telemetry_files))

## External input audit

For the files supplied for this revision, the expected high-level inventory is:

- **CICIoT2023 (`IoT_Intrusion.csv`)**: 47 features with benign and multiple IoT attack classes. The exact local file hash and record count are retained in audit outputs rather than emphasized in figures or manuscript-facing labels.
- **ToN-IoT `train_test_network.csv`**: approximately 211,043 rows, 44 columns, binary `label` plus attack `type`.
- **ToN-IoT telemetry files**: seven device/service-specific CSVs. These are catalogued but excluded from the flow-level cross-domain feature mapping.

The code below recomputes the audit from the actual files and records SHA-256 hashes for reproducibility.

In [ ]:
def count_csv_rows(path):
    """Count CSV data rows without relying on a later helper cell."""
    with Path(path).open("rb") as f:
        return max(sum(1 for _ in f) - 1, 0)

def external_file_audit(path, dataset_name, label_col=None):
    header = pd.read_csv(path, nrows=0)
    row_count = count_csv_rows(path)
    out = {
        "dataset": dataset_name,
        "file": path.name,
        "rows": row_count,
        "columns": len(header.columns),
        "sha256": sha256_file(path),
        "size_mb": path.stat().st_size / 1024**2,
    }
    if label_col and label_col in header.columns:
        vc = {}
        for chunk in pd.read_csv(path, usecols=[label_col], chunksize=200_000):
            for k, v in chunk[label_col].astype(str).value_counts().items():
                vc[k] = vc.get(k, 0) + int(v)
        out["unique_labels"] = len(vc)
        out["label_counts_json"] = json.dumps(vc, sort_keys=True)
    return out

audit_rows = [
    external_file_audit(CIC_FILE, "CICIoT2023", "label"),
    external_file_audit(TON_NETWORK_FILE, "ToN-IoT Network", "label"),
]
for name, path in telemetry_files.items():
    audit_rows.append(external_file_audit(path, f"ToN-IoT Telemetry — {name}", "label"))

input_audit = pd.DataFrame(audit_rows)
input_audit.to_csv(OUT_DIR / "audit" / "external_input_inventory.csv", index=False)
input_audit

In [ ]:
# Supplied-file fingerprint sanity checks (warnings only; SHA-256 values are recorded for exact provenance).
expected_inventory = {
    "CICIoT2023": {"rows":1048575, "columns":47},
    "ToN-IoT Network": {"rows":211043, "columns":44},
}
for dataset_name, expected in expected_inventory.items():
    row = input_audit[input_audit["dataset"]==dataset_name]
    if len(row):
        observed_rows=int(row.iloc[0]["rows"])
        observed_cols=int(row.iloc[0]["columns"])
        if observed_rows!=expected.get("rows", observed_rows) or observed_cols!=expected["columns"]:
            print(
                f"[WARN] {dataset_name}: supplied file differs from the revision file inspected when this notebook "
                f"was authored. Observed rows/columns={observed_rows}/{observed_cols}; "
                f"expected columns={expected['columns']}. "
                "The analysis may still proceed, but document the exact SHA-256 and inventory."
            )
        else:
            print(f"[PASS] {dataset_name}: row/column inventory matches the inspected revision file.")

In [ ]:
# Inspect external labels without loading unnecessary columns.
cic_label_counts = pd.read_csv(CIC_FILE, usecols=["label"])["label"].value_counts()
ton_label_counts = pd.read_csv(TON_NETWORK_FILE, usecols=["label", "type"])

print("CIC rows:", int(cic_label_counts.sum()))
print("CIC labels:", len(cic_label_counts))
print("CIC benign:", int(cic_label_counts.get("BenignTraffic", 0)))
print("CIC attacks:", int(cic_label_counts.sum() - cic_label_counts.get("BenignTraffic", 0)))

print("\nToN Network rows:", len(ton_label_counts))
print(ton_label_counts["label"].value_counts().sort_index())
print("\nToN attack types:")
print(ton_label_counts["type"].value_counts())

In [ ]:
# ---------------------------------------------------------------------------
# External-data provenance.
#
# CICIoT2023 naming follows the source dataset page for `IoT_Intrusion.csv`,
# which attributes the dataset to the CICIoT2023 study. Exact local filename,
# record count, and SHA-256 are retained in the audit for reproducibility, but
# manuscript-facing text and figures use the canonical dataset name CICIoT2023.
# ---------------------------------------------------------------------------
def count_csv_data_rows(path: Path) -> int:
    """Count CSV data records without loading the full file into memory."""
    with Path(path).open("rb") as f:
        return max(sum(1 for _ in f) - 1, 0)

cic_rows = count_csv_data_rows(CIC_FILE)
ton_rows = count_csv_data_rows(TON_NETWORK_FILE)
CIC_DESCRIPTION = "CICIoT2023"

provenance = pd.DataFrame([
    {
        "dataset": "CICIoT2023",
        "file": CIC_FILE.name,
        "records": int(cic_rows),
        "manuscript_description": "CICIoT2023",
        "source_page_name": "IoT Intrusion Detection",
        "source_file_name": CIC_FILE.name,
        "source_url": "https://www.kaggle.com/datasets/subhajournal/iotintrusion",
        "feature_semantics_caveat":
            "CICIoT2023 features are harmonized approximately with HITL-IoT; "
            "the shared-feature analysis does not assume one-to-one feature equivalence.",
        "sha256": sha256_file(CIC_FILE),
    },
    {
        "dataset": "ToN-IoT Network",
        "file": TON_NETWORK_FILE.name,
        "records": int(ton_rows),
        "manuscript_description": "ToN-IoT Network",
        "source_page_name": "ToN-IoT",
        "source_file_name": TON_NETWORK_FILE.name,
        "source_url": "https://research.unsw.edu.au/projects/toniot-datasets",
        "feature_semantics_caveat":
            "ToN-IoT network features are harmonized approximately with HITL-IoT; "
            "the shared-feature analysis does not assume one-to-one feature equivalence.",
        "sha256": sha256_file(TON_NETWORK_FILE),
    },
])

provenance.to_csv(
    OUT_DIR / "audit" / "external_dataset_provenance.csv",
    index=False,
)

print("[PASS] CICIoT2023 source identity recorded; exact local file hash retained.")
print("[INFO] CICIoT2023 manuscript label:", CIC_DESCRIPTION)
display(provenance.drop(columns=["records"]))

# 1. Semantic Feature Harmonization

Cross-dataset validation is only meaningful when variables are mapped by **measurement semantics**, not by similar-looking names.

### Conservative universal core

The primary three-domain comparison uses:

| Harmonized feature | HITL-IoT | CICIoT2023 | ToN-IoT Network | Mapping note |
|---|---|---|---|---|
| `duration_s` | `duration` | `flow_duration` | `duration` | Direct flow-duration concept |
| `packet_count` | `packet_count` | `Number` | `src_pkts + dst_pkts` | CIC paper describes `Number` as packets in flow |
| `total_bytes` | `bytes_sent + bytes_received` | `Tot sum` | `src_bytes + dst_bytes` | Total traffic volume; direction collapsed |
| `mean_packet_size` | `packet_size` | `AVG` | `total_bytes / total_packets` | Semantically approximate mean size |
| `proto_tcp` | `protocol` | `TCP` | `proto` | Transport family |
| `proto_udp` | `protocol` | `UDP` | `proto` | Transport family |
| `proto_other` | derived | complement of TCP/UDP | complement of TCP/UDP | Preserves non-TCP/UDP external traffic |

### Pairwise enriched sensitivity

**HITL ↔ CICIoT2023** additionally uses `packet_rate`. `ttl` is deliberately excluded because HITL-IoT TTL is a per-device constant by design.

**HITL ↔ ToN-IoT Network** additionally uses three broad destination-port bands: `dst_port_well_known` (0–1023), `dst_port_registered` (1024–49151), and `dst_port_dynamic` (49152–65535). This avoids treating port identifiers as continuous magnitudes.

Features with questionable directional equivalence, application payload semantics, generated context, profile metadata, or target-derived information are deliberately excluded.

All cross-domain mappings are treated as **semantically harmonized approximations**. In particular, CICIoT2023 packet-window aggregates are not described as one-to-one equivalents of HITL-IoT flow totals.

In [ ]:
# Required columns only: this keeps memory use modest for the CICIoT2023 source file.
HITL_USECOLS = [
    "flow_id", "timestamp", "duration", "packet_rate", "packet_count",
    "packet_size", "bytes_sent", "bytes_received", "protocol", "dst_port",
    "ttl", "is_attack"
]
CIC_USECOLS = [
    "flow_duration", "Rate", "Number", "AVG", "Tot sum", "Duration",
    "TCP", "UDP", "ICMP", "label"
]
TON_USECOLS = [
    "ts", "duration", "src_bytes", "dst_bytes", "src_pkts", "dst_pkts",
    "proto", "dst_port", "label", "type"
]

hitl_raw = pd.read_csv(HITL_RELEASE / "HITL-IoT_dataset_v1.1.csv", usecols=HITL_USECOLS)
cic_raw = pd.read_csv(CIC_FILE, usecols=CIC_USECOLS)
ton_raw = pd.read_csv(TON_NETWORK_FILE, usecols=lambda c: c in TON_USECOLS)  # tolerant if ts is absent

hitl_raw["timestamp"] = pd.to_datetime(hitl_raw["timestamp"])

print("Loaded:", len(hitl_raw), len(cic_raw), len(ton_raw))

In [ ]:
def finite_nonnegative(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    x = x.replace([np.inf, -np.inf], np.nan)
    return x.where(x >= 0)

def add_port_bands(z, port_series):
    """Encode destination port as broad IANA-style numeric ranges."""
    p = finite_nonnegative(port_series)
    valid = p.between(0, 65535, inclusive="both")
    z["dst_port_well_known"] = (valid & p.between(0, 1023, inclusive="both")).astype(int)
    z["dst_port_registered"] = (valid & p.between(1024, 49151, inclusive="both")).astype(int)
    z["dst_port_dynamic"] = (valid & p.between(49152, 65535, inclusive="both")).astype(int)
    return z

def harmonize_hitl(df):
    z = pd.DataFrame(index=df.index)
    z["row_id"] = df["flow_id"].astype(str)
    z["timestamp"] = df["timestamp"]
    z["y"] = df["is_attack"].astype(int)
    z["duration_s"] = finite_nonnegative(df["duration"])
    z["packet_rate"] = finite_nonnegative(df["packet_rate"])
    z["packet_count"] = finite_nonnegative(df["packet_count"])
    z["mean_packet_size"] = finite_nonnegative(df["packet_size"])
    z["total_bytes"] = finite_nonnegative(df["bytes_sent"] + df["bytes_received"])
    # Retained for audit only; excluded from every model.
    z["ttl"] = finite_nonnegative(df["ttl"])
    z = add_port_bands(z, df["dst_port"])
    proto = df["protocol"].astype(str).str.upper().str.strip()
    z["proto_tcp"] = (proto == "TCP").astype(int)
    z["proto_udp"] = (proto == "UDP").astype(int)
    z["proto_other"] = (~proto.isin(["TCP", "UDP"])).astype(int)
    z["domain"] = "HITL-IoT"
    return z

def harmonize_cic(df):
    z = pd.DataFrame(index=df.index)
    z["row_id"] = "CIC-" + pd.Series(np.arange(len(df)), index=df.index).astype(str)
    z["timestamp"] = pd.NaT
    z["y"] = (df["label"].astype(str) != "BenignTraffic").astype(int)
    z["duration_s"] = finite_nonnegative(df["flow_duration"])
    z["packet_rate"] = finite_nonnegative(df["Rate"])
    z["packet_count"] = finite_nonnegative(df["Number"])
    z["mean_packet_size"] = finite_nonnegative(df["AVG"])
    z["total_bytes"] = finite_nonnegative(df["Tot sum"])
    # Loaded for provenance/semantic audit only; excluded from all model feature sets.
    z["ttl"] = finite_nonnegative(df["Duration"])
    tcp = pd.to_numeric(df["TCP"], errors="coerce").fillna(0) > 0
    udp = pd.to_numeric(df["UDP"], errors="coerce").fillna(0) > 0
    z["proto_tcp"] = tcp.astype(int)
    z["proto_udp"] = udp.astype(int)
    z["proto_other"] = (~(tcp | udp)).astype(int)
    z["domain"] = "CICIoT2023"
    return z

def harmonize_ton(df):
    z = pd.DataFrame(index=df.index)
    z["row_id"] = "TON-" + pd.Series(np.arange(len(df)), index=df.index).astype(str)

    if "ts" in df.columns:
        ts_num = pd.to_numeric(df["ts"], errors="coerce")
        z["timestamp"] = pd.to_datetime(ts_num, unit="s", errors="coerce")
    else:
        z["timestamp"] = pd.NaT

    z["y"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    z["duration_s"] = finite_nonnegative(df["duration"])
    total_packets = finite_nonnegative(df["src_pkts"]) + finite_nonnegative(df["dst_pkts"])
    total_bytes = finite_nonnegative(df["src_bytes"]) + finite_nonnegative(df["dst_bytes"])
    z["packet_count"] = total_packets
    z["total_bytes"] = total_bytes
    z["mean_packet_size"] = np.where(total_packets > 0, total_bytes / total_packets, np.nan)
    z["packet_rate"] = np.nan  # Not used in the universal map.
    z["ttl"] = np.nan
    z = add_port_bands(z, df["dst_port"])
    proto = df["proto"].astype(str).str.lower().str.strip()
    z["proto_tcp"] = (proto == "tcp").astype(int)
    z["proto_udp"] = (proto == "udp").astype(int)
    z["proto_other"] = (~proto.isin(["tcp", "udp"])).astype(int)
    z["domain"] = "ToN-IoT"
    return z

hitl = harmonize_hitl(hitl_raw)
cic = harmonize_cic(cic_raw)
ton = harmonize_ton(ton_raw)

UNIVERSAL_FEATURES = [
    "duration_s", "packet_count", "total_bytes", "mean_packet_size",
    "proto_tcp", "proto_udp", "proto_other"
]
CIC_PAIR_FEATURES = UNIVERSAL_FEATURES + ["packet_rate"]
TON_PORT_BAND_FEATURES = [
    "dst_port_well_known", "dst_port_registered", "dst_port_dynamic"
]
TON_PAIR_FEATURES = UNIVERSAL_FEATURES + TON_PORT_BAND_FEATURES

assert set(UNIVERSAL_FEATURES) <= set(hitl.columns) & set(cic.columns) & set(ton.columns)
assert set(TON_PORT_BAND_FEATURES) <= set(hitl.columns) & set(ton.columns)

TON_HAS_USABLE_TIMESTAMP = bool(
    ("ts" in ton_raw.columns) and (ton["timestamp"].notna().mean() > 0.99)
)

print("Universal features:", UNIVERSAL_FEATURES)
print("HITL-CIC enriched:", CIC_PAIR_FEATURES)
print("HITL-ToN enriched:", TON_PAIR_FEATURES)
print(
    "[INFO] ToN timestamp availability:",
    "usable" if TON_HAS_USABLE_TIMESTAMP else "not available in supplied network file"
)

In [ ]:
mapping_rows = [
    ["duration_s", "duration", "flow_duration", "duration", "High",
     "Direct duration concept, but source collection units/procedures may differ"],
    ["packet_count", "packet_count", "Number", "src_pkts + dst_pkts", "Medium",
     "CIC Number is a packet-window aggregate, not assumed to be a per-flow total"],
    ["total_bytes", "bytes_sent + bytes_received", "Tot sum", "src_bytes + dst_bytes", "Medium",
     "Direction collapsed; CIC Tot sum is a packet-window aggregate and ToN byte semantics differ from generated HITL totals"],
    ["mean_packet_size", "packet_size", "AVG", "total_bytes / total_packets", "Medium",
     "Approximate mean packet-size concept"],
    ["proto_tcp", "protocol == TCP", "TCP", "proto == tcp", "High",
     "Transport family"],
    ["proto_udp", "protocol == UDP", "UDP", "proto == udp", "High",
     "Transport family"],
    ["proto_other", "derived", "not TCP/UDP", "not tcp/udp", "Medium",
     "Preserves non-TCP/UDP traffic without forcing protocol-name equivalence"],
    ["packet_rate", "packet_rate", "Rate", "not primary", "Medium–High (HITL-CIC)",
     "Semantically similar rate variable; source aggregation windows differ"],
    ["ttl", "ttl", "Duration", "not available", "Excluded",
     "CIC Duration is documented as TTL, but HITL ttl is a per-device constant by design; excluded from every model"],
    ["dst_port_well_known", "dst_port 0–1023", "not available", "dst_port 0–1023", "High (HITL-ToN)",
     "Broad port-category indicator; avoids treating port identifiers as continuous magnitudes"],
    ["dst_port_registered", "dst_port 1024–49151", "not available", "dst_port 1024–49151", "High (HITL-ToN)",
     "Broad port-category indicator"],
    ["dst_port_dynamic", "dst_port 49152–65535", "not available", "dst_port 49152–65535", "High (HITL-ToN)",
     "Broad port-category indicator"],
]
feature_mapping = pd.DataFrame(mapping_rows, columns=[
    "harmonized_feature", "HITL-IoT", "CICIoT2023", "ToN-IoT",
    "mapping_strength", "note"
])
feature_mapping.to_csv(
    OUT_DIR / "tables" / "Table_External_Feature_Mapping.csv", index=False
)
feature_mapping

# 2. Frozen HITL temporal split and external 70/15/15 splits

The frozen Step-6 validation/test probability files identify the exact HITL temporal validation and test flow IDs. All remaining HITL flows form the temporal training partition. This avoids relying on a separate IID split file.

CICIoT2023 and ToN-IoT are split internally using fixed-seed stratification only to create source-domain training/validation/test partitions for the external-validation experiment.

In [ ]:
p_val_path = STEP6_DIR / "probabilities" / "temporal_validation_probabilities_all9.csv"
p_test_path = STEP6_DIR / "probabilities" / "temporal_test_probabilities_all9.csv"

p_val_ids = set(pd.read_csv(p_val_path, usecols=["flow_id"])["flow_id"].unique())
p_test_ids = set(pd.read_csv(p_test_path, usecols=["flow_id"])["flow_id"].unique())

hitl["split"] = np.where(
    hitl["row_id"].isin(p_val_ids), "validation",
    np.where(hitl["row_id"].isin(p_test_ids), "test", "train")
)

assert hitl.loc[hitl.split=="train", "timestamp"].max() < hitl.loc[hitl.split=="validation", "timestamp"].min()
assert hitl.loc[hitl.split=="validation", "timestamp"].max() < hitl.loc[hitl.split=="test", "timestamp"].min()

def add_external_split(df, seed=SEED):
    train_idx, temp_idx = train_test_split(
        np.arange(len(df)), test_size=0.30, random_state=seed, stratify=df["y"]
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=seed, stratify=df.iloc[temp_idx]["y"]
    )
    out = df.copy()
    out["split"] = ""
    out.iloc[train_idx, out.columns.get_loc("split")] = "train"
    out.iloc[val_idx, out.columns.get_loc("split")] = "validation"
    out.iloc[test_idx, out.columns.get_loc("split")] = "test"
    return out

cic = add_external_split(cic)
ton = add_external_split(ton)

split_summary = pd.concat([
    hitl.groupby(["domain","split"])["y"].agg(["size","mean"]).reset_index(),
    cic.groupby(["domain","split"])["y"].agg(["size","mean"]).reset_index(),
    ton.groupby(["domain","split"])["y"].agg(["size","mean"]).reset_index(),
], ignore_index=True).rename(columns={"mean":"attack_rate"})
split_summary.to_csv(OUT_DIR / "tables" / "Table_External_Splits.csv", index=False)
split_summary

In [ ]:
# ---------------------------------------------------------------------------
# Duplicate and cross-partition leakage audit.
#
# A random split can place the same harmonized feature vector in development
# and test, inflating in-domain results. We therefore report:
#   (1) overlap with training, and
#   (2) a stricter "development-novel" test subset whose feature vector occurs
#       in neither training nor validation.
# ---------------------------------------------------------------------------
def vector_hash(df, features=UNIVERSAL_FEATURES):
    return pd.util.hash_pandas_object(
        df[features].round(9).fillna(-1.0), index=False
    ).to_numpy()

dup_rows = []
NOVEL_TEST_IDS = {}

for name, d in {"HITL-IoT": hitl, "CICIoT2023": cic, "ToN-IoT": ton}.items():
    h = vector_hash(d)
    hs = pd.Series(h, index=d.index)

    dup_any = hs.duplicated(keep=False)

    conflict = (
        pd.DataFrame({"h": h, "y": d["y"].to_numpy()})
        .groupby("h")["y"].nunique()
    )
    conflicting = hs.map(conflict).gt(1)

    train_mask = d["split"] == "train"
    val_mask = d["split"] == "validation"
    test_mask = d["split"] == "test"

    train_h = set(hs[train_mask])
    development_h = set(hs[train_mask]) | set(hs[val_mask])

    test_h = hs[test_mask]
    test_in_train = test_h.isin(train_h)
    test_in_development = test_h.isin(development_h)

    # Strict robustness set used throughout later duplicate-sensitive analyses.
    strict_novel_ids = set(
        d.loc[test_mask, "row_id"][~test_in_development.to_numpy()]
    )
    NOVEL_TEST_IDS[name] = strict_novel_ids

    novel_y = d.loc[d["row_id"].isin(strict_novel_ids), "y"]

    dup_rows.append({
        "domain": name,
        "rows": len(d),
        "unique_feature_vectors": int(hs.nunique()),
        "rows_in_duplicated_vectors_pct": 100 * dup_any.mean(),
        "rows_with_label_conflicting_vectors_pct": 100 * conflicting.mean(),
        "test_rows": int(test_mask.sum()),
        "test_vectors_also_in_train_pct": 100 * test_in_train.mean(),
        "test_vectors_also_in_development_pct": 100 * test_in_development.mean(),
        "development_novel_test_rows": len(strict_novel_ids),
        "development_novel_test_attack_rate":
            float(novel_y.mean()) if len(novel_y) else np.nan,
    })

duplicate_audit = pd.DataFrame(dup_rows)
duplicate_audit.to_csv(
    OUT_DIR / "audit" / "external_duplicate_leakage_audit.csv", index=False
)

for r in dup_rows:
    print(
        f"{r['domain']:12s}: "
        f"{r['rows_in_duplicated_vectors_pct']:5.1f}% rows in duplicated vectors; "
        f"{r['test_vectors_also_in_train_pct']:5.1f}% test vectors occur in train; "
        f"{r['test_vectors_also_in_development_pct']:5.1f}% occur in train/validation; "
        f"development-novel test rows={r['development_novel_test_rows']:,}"
    )

duplicate_audit

# 3. Distributional external validation

To reduce prevalence confounding, primary distribution comparisons are **class-conditional**:

- HITL benign vs external benign
- HITL attack vs external attack

Magnitude variables are compared on `log1p` scale because flow duration, volume, and packet counts are heavy-tailed.

Metrics:

- **KS statistic:** maximum empirical-CDF separation.
- **Jensen–Shannon divergence:** bounded distribution-shape divergence.
- **Normalized Wasserstein distance:** Earth-mover distance divided by pooled IQR.
- **Correlation-matrix distance:** normalized Frobenius distance between feature-correlation matrices.
- **Domain-discriminator AUROC:** how easily a classifier distinguishes datasets after balancing both attack class and domain sample size. Values near 0.5 indicate overlap; high values indicate strong domain separation.

These metrics characterize domain correspondence. They are not combined into a single "realism score."

In [ ]:
MAGNITUDE_FEATURES = ["duration_s", "packet_count", "total_bytes", "mean_packet_size"]

def balanced_sample(df, per_class, seed=SEED):
    parts = []
    for y in [0,1]:
        g = df[df["y"]==y]
        n = min(len(g), per_class)
        parts.append(g.sample(n=n, random_state=seed + y))
    return pd.concat(parts, ignore_index=True)

def analysis_transform(x, feature):
    x = pd.to_numeric(pd.Series(x), errors="coerce").replace([np.inf,-np.inf], np.nan).dropna().to_numpy(float)
    if feature in MAGNITUDE_FEATURES or feature in ["packet_rate", "ttl", "dst_port"]:
        x = np.log1p(np.clip(x, 0, None))
    return x

def js_divergence(a, b, bins=50):
    a = np.asarray(a,float); b = np.asarray(b,float)
    pooled = np.r_[a,b]
    lo, hi = np.quantile(pooled, [0.005,0.995])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return 0.0
    edges = np.linspace(lo, hi, bins+1)
    pa, _ = np.histogram(np.clip(a,lo,hi), bins=edges)
    pb, _ = np.histogram(np.clip(b,lo,hi), bins=edges)
    pa = pa.astype(float) + 1e-12
    pb = pb.astype(float) + 1e-12
    pa /= pa.sum(); pb /= pb.sum()
    return float(jensenshannon(pa,pb,base=2.0)**2)

def normalized_wasserstein(a, b):
    a = np.asarray(a,float); b = np.asarray(b,float)
    pooled = np.r_[a,b]
    q25,q75 = np.quantile(pooled,[.25,.75])
    scale = q75-q25
    if scale <= 1e-12:
        scale = np.std(pooled)
    if scale <= 1e-12:
        return 0.0
    return float(wasserstein_distance(a,b)/scale)

def pair_distribution_metrics(a_df, b_df, pair_name, features=UNIVERSAL_FEATURES):
    rows=[]
    for y, cname in [(0,"benign"),(1,"attack")]:
        aa = a_df[a_df["y"]==y]
        bb = b_df[b_df["y"]==y]
        for f in features:
            a = analysis_transform(aa[f], f)
            b = analysis_transform(bb[f], f)
            if len(a)==0 or len(b)==0:
                continue
            rows.append({
                "pair":pair_name, "class":cname, "feature":f,
                "n_a":len(a), "n_b":len(b),
                "ks":float(ks_2samp(a,b).statistic),
                "jsd":js_divergence(a,b),
                "normalized_wasserstein":normalized_wasserstein(a,b),
            })
    return pd.DataFrame(rows)

hitl_dist = balanced_sample(hitl, DIST_SAMPLE_PER_CLASS)
cic_dist = balanced_sample(cic, DIST_SAMPLE_PER_CLASS)
ton_dist = balanced_sample(ton, DIST_SAMPLE_PER_CLASS)

dist_metrics = pd.concat([
    pair_distribution_metrics(hitl_dist,cic_dist,"HITL-IoT vs CICIoT2023"),
    pair_distribution_metrics(hitl_dist,ton_dist,"HITL-IoT vs ToN-IoT"),
], ignore_index=True)

dist_metrics.to_csv(OUT_DIR/"tables"/"Table_External_Distribution_Distances.csv",index=False)
dist_metrics.head(20)

In [ ]:
def transformed_matrix(df, features=UNIVERSAL_FEATURES):
    z = pd.DataFrame(index=df.index)
    for f in features:
        x = pd.to_numeric(df[f], errors="coerce").replace([np.inf,-np.inf],np.nan)
        if f in MAGNITUDE_FEATURES:
            x = np.log1p(x.clip(lower=0))
        z[f]=x
    return z.fillna(z.median(numeric_only=True)).fillna(0)

def corr_distance(a,b,features=UNIVERSAL_FEATURES):
    ca=transformed_matrix(a,features).corr().fillna(0).to_numpy()
    cb=transformed_matrix(b,features).corr().fillna(0).to_numpy()
    return float(np.linalg.norm(ca-cb,"fro")/math.sqrt(ca.size))

corr_rows=[
    {"pair":"HITL-IoT vs CICIoT2023","correlation_matrix_distance":corr_distance(hitl_dist,cic_dist)},
    {"pair":"HITL-IoT vs ToN-IoT","correlation_matrix_distance":corr_distance(hitl_dist,ton_dist)},
]
corr_table=pd.DataFrame(corr_rows)
corr_table.to_csv(OUT_DIR/"tables"/"Table_External_Correlation_Distance.csv",index=False)
corr_table

In [ ]:
# Balanced domain discriminator with duplicate-robust source preparation.
#
# Exact harmonized vectors are collapsed within each domain BEFORE the
# train/test split. This prevents the nonlinear discriminator from benefiting
# merely because the same repeated vector occurs in both discriminator
# training and test partitions.
def deduplicate_domain_vectors(df, features=UNIVERSAL_FEATURES):
    z = df.copy()
    z["_vector_hash"] = vector_hash(z, features)

    # Resolve rare label-conflicting vectors deterministically by majority class
    # only for the purpose of class-balanced domain-discriminator sampling.
    majority_y = (
        z.groupby("_vector_hash")["y"]
        .mean()
        .ge(0.5)
        .astype(int)
    )
    z = z.drop_duplicates("_vector_hash", keep="first").copy()
    z["y"] = z["_vector_hash"].map(majority_y).astype(int)
    return z.drop(columns="_vector_hash")

def domain_discriminator_auc(a_df, b_df, name, per_class=DOMAIN_CLASSIFIER_PER_CLASS):
    a_unique = deduplicate_domain_vectors(a_df)
    b_unique = deduplicate_domain_vectors(b_df)

    aa = balanced_sample(a_unique, per_class)
    bb = balanced_sample(b_unique, per_class)
    aa = aa.copy(); bb = bb.copy()
    aa["domain_target"] = 0
    bb["domain_target"] = 1

    pooled = pd.concat([aa, bb], ignore_index=True)
    X = transformed_matrix(pooled, UNIVERSAL_FEATURES)
    y = pooled["domain_target"].to_numpy()

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=.30, random_state=SEED, stratify=y
    )

    lr = Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=SEED
        ))
    ])
    lr.fit(Xtr, ytr)
    p_lr = lr.predict_proba(Xte)[:, 1]

    rf = RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced_subsample",
    )
    rf.fit(Xtr, ytr)
    p_rf = rf.predict_proba(Xte)[:, 1]

    return {
        "pair": name,
        "domain_classifier_auroc": roc_auc_score(yte, p_lr),
        "domain_classifier_auroc_rf": roc_auc_score(yte, p_rf),
        "deduplicated_before_split": True,
        "unique_vectors_domain_a": len(a_unique),
        "unique_vectors_domain_b": len(b_unique),
        "balanced_sample_n": len(pooled),
    }

domain_auc = pd.DataFrame([
    domain_discriminator_auc(hitl, cic, "HITL-IoT vs CICIoT2023"),
    domain_discriminator_auc(hitl, ton, "HITL-IoT vs ToN-IoT"),
])
domain_auc.to_csv(
    OUT_DIR / "tables" / "Table_External_Domain_Discriminator.csv",
    index=False
)
domain_auc

In [ ]:
# PCA visualization using class-balanced samples from all three domains.
pca_parts=[]
for d in [hitl,cic,ton]:
    s=balanced_sample(d,PCA_SAMPLE_PER_CLASS).copy()
    pca_parts.append(s)
pca_df=pd.concat(pca_parts,ignore_index=True)

Xp=transformed_matrix(pca_df,UNIVERSAL_FEATURES)
scaler=StandardScaler()
Xpz=scaler.fit_transform(Xp)
pca=PCA(n_components=2,random_state=SEED)
coords=pca.fit_transform(Xpz)
pca_plot=pca_df[["domain","y"]].copy()
pca_plot["PC1"]=coords[:,0]
pca_plot["PC2"]=coords[:,1]
pca_plot.to_csv(OUT_DIR/"tables"/"External_PCA_Coordinates_Sample.csv",index=False)

print("Explained variance:",pca.explained_variance_ratio_)

In [ ]:
# Figure 13 — External-domain characterisation
fig = plt.figure(figsize=(DOUBLE_COL, 6.1))
gs = fig.add_gridspec(2, 2, hspace=0.52, wspace=0.55, width_ratios=[1.0, 1.15])

ax = fig.add_subplot(gs[0, 0])
yy = np.arange(3)[::-1]
for y_, d in zip(yy, [hitl, cic, ton]):
    a = float(d["y"].mean())
    ax.barh(y_, 1 - a, color=BENIGN_C, height=0.6, edgecolor="white")
    ax.barh(y_, a, left=1 - a, color=ATTACK_C, height=0.6, edgecolor="white", hatch="////")
    ax.text(0.985, y_, f"{100*a:.1f}%", va="center", ha="right", fontsize=7, color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.18", facecolor=ATTACK_C, edgecolor="none"))
    ax.text(0.015, y_, f"{100*(1-a):.1f}%", va="center", ha="left", fontsize=7, color="white", fontweight="bold") if (1 - a) > 0.12 else None
ax.set_yticks(yy, DOMAIN_ORDER, fontsize=7.5)
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel("Share of records"); ax.tick_params(axis="y", length=0)
ax.legend(handles=[mpl.patches.Patch(color=BENIGN_C, label="Benign"),
                   mpl.patches.Patch(facecolor=ATTACK_C, hatch="////", edgecolor="white", label="Attack")],
          loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=2)
add_panel_label(ax, "a", x=-0.42, y=1.12)

FEAT_LABEL = {"duration_s": "Duration", "packet_count": "Packet count", "total_bytes": "Total bytes",
              "mean_packet_size": "Mean packet size", "proto_tcp": "TCP", "proto_udp": "UDP", "proto_other": "Other protocol"}
ax = fig.add_subplot(gs[0, 1])
heat = dist_metrics.pivot_table(index="feature", columns=["pair", "class"], values="jsd").reindex(UNIVERSAL_FEATURES)
cols = [(p, c) for p in ["HITL-IoT vs CICIoT2023", "HITL-IoT vs ToN-IoT"] for c in ["benign", "attack"] if (p, c) in heat.columns]
M = heat[cols].to_numpy()
im = ax.imshow(M, cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if np.isfinite(M[i, j]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.6, color="white" if M[i, j] > 0.55 else "#1A1A1A")
ax.set_xticks(range(len(cols)), [f"{p.split(' vs ')[1]}\n{c}" for p, c in cols], fontsize=7)
ax.set_yticks(range(len(UNIVERSAL_FEATURES)), [FEAT_LABEL.get(f, f) for f in UNIVERSAL_FEATURES], fontsize=7)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03); cb.set_label("JSD (base 2)", fontsize=7); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=6.5, length=2)
add_panel_label(ax, "b", x=-0.48, y=1.04)

ax = fig.add_subplot(gs[1, 0])
for dom in DOMAIN_ORDER:
    g = pca_plot[pca_plot.domain == dom]
    ax.scatter(g.PC1, g.PC2, s=3, alpha=0.22, color=DOMAIN_COLORS[dom], marker=DOMAIN_MARKERS[dom], linewidths=0, rasterized=True)
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.0f}% variance)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.0f}% variance)")
ax.legend(handles=[Line2D([], [], marker=DOMAIN_MARKERS[d], ls="", color=DOMAIN_COLORS[d], ms=5, label=d) for d in DOMAIN_ORDER],
          loc="best", fontsize=7)
ygrid(ax, "both"); add_panel_label(ax, "c", x=-0.22)

ax = fig.add_subplot(gs[1, 1])
yy = np.arange(len(domain_auc))[::-1]
for col, mk, lbl in (("domain_classifier_auroc", "o", "Logistic regression (lower bound)"),
                     ("domain_classifier_auroc_rf", "s", "Random forest (non-linear)")):
    ax.plot(domain_auc[col], yy, mk, ms=6, color=MODEL_COLORS["Logistic Regression" if mk == "o" else "Random Forest"],
            label=lbl, ls="", mec="white", mew=0.5, zorder=3)
    for y_, v in zip(yy, domain_auc[col]):
        ax.text(v, y_ + (0.18 if mk == "o" else -0.22), f"{v:.3f}", ha="center", fontsize=6.6, color=NEUTRAL)
ax.axvline(0.5, ls="--", lw=0.8, color="#9A9A9A"); ax.text(0.505, yy[-1] - 0.45, "indistinguishable", fontsize=6.6, color=NEUTRAL)
ax.set_xlim(0.45, 1.03); ax.set_ylim(-0.8, len(yy) - 0.3)
ax.set_yticks(yy, domain_auc["pair"], fontsize=7.5)
ax.set_xlabel("Domain-discriminator AUROC"); ax.tick_params(axis="y", length=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.24), ncol=1, fontsize=7)
ygrid(ax, "x"); add_panel_label(ax, "d", x=-0.48)

CAPTION_FIG13 = 'Figure 13. External-domain characterisation of HITL-IoT v1.1 against CICIoT2023 and ToN-IoT. (a) Native class prevalence. (b) Class-conditional Jensen–Shannon divergence (JSD, base 2; 0 = identical, 1 = disjoint) per harmonised feature on class-balanced samples. (c) Principal-component projection of the harmonised features. (d) Balanced domain-discriminator AUROC after exact-vector deduplication within each domain. Feature semantics are harmonised approximations; CICIoT2023 features are packet-window aggregates.'
save_publication_figure(fig, "Figure13_External_Domain_Characterization", CAPTION_FIG13, 4)
plt.show(); plt.close(fig); print(CAPTION_FIG13)


# 4. Cross-domain transfer

The primary transfer matrix uses the **same universal feature set for all three domains**, enabling a true 3×3 comparison.

For each source domain:

1. Fit the model on source `train`.
2. Choose the binary decision threshold on source `validation` by Macro-F1.
3. Freeze the model and threshold.
4. Evaluate on the held-out `test` partition of **every** domain.

No target labels influence model fitting or threshold selection.

Two fixed model families are evaluated:

- Logistic Regression — linear reference.
- Random Forest — nonlinear reference aligned with the Step-6/Step-7 selected family.

Pairwise enriched feature mappings are run separately as sensitivity analyses.

In [ ]:
BINARY_FEATURES=["proto_tcp","proto_udp","proto_other","dst_port_well_known","dst_port_registered","dst_port_dynamic"]

def make_model_pipeline(model_name, features):
    numeric=[f for f in features if f not in BINARY_FEATURES]
    pre=ColumnTransformer([
        ("mag",Pipeline([
            ("impute",SimpleImputer(strategy="median")),
            ("log1p",FunctionTransformer(np.log1p, validate=False)),
            ("scale",RobustScaler(quantile_range=(5,95))),
        ]),numeric),
        ("bin",Pipeline([
            ("impute",SimpleImputer(strategy="most_frequent")),
        ]),[f for f in features if f in BINARY_FEATURES]),
    ],remainder="drop")
    if model_name=="Logistic Regression":
        clf=LogisticRegression(C=1.0,class_weight="balanced",max_iter=4000,solver="lbfgs",random_state=SEED)
    elif model_name=="Random Forest":
        clf=RandomForestClassifier(
            n_estimators=RF_TREES,max_depth=None,min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            class_weight="balanced_subsample",n_jobs=-1,random_state=SEED
        )
    else:
        raise ValueError(model_name)
    return Pipeline([("pre",pre),("clf",clf)])

def maybe_cap_train(df, domain_name):
    if not FAST_MODE or domain_name!="CICIoT2023" or len(df)<=FAST_CIC_TRAIN_CAP:
        return df
    _, capped = train_test_split(
        df, test_size=FAST_CIC_TRAIN_CAP, random_state=SEED, stratify=df["y"]
    )
    return capped

def safe_auc(y,p):
    return roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan

def safe_ap(y,p):
    return average_precision_score(y,p) if len(np.unique(y))>1 else np.nan

def metrics_at_threshold(y,p,t):
    y=np.asarray(y,int); p=np.asarray(p,float); pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "accuracy":accuracy_score(y,pred),
        "balanced_accuracy":balanced_accuracy_score(y,pred),
        "precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),
        "f1":f1_score(y,pred,zero_division=0),
        "macro_f1":f1_score(y,pred,average="macro",zero_division=0),
        # undefined when only one class is present or predicted: NaN, not a misleading 0.0
        "mcc":matthews_corrcoef(y,pred) if (len(np.unique(pred))>1 and len(np.unique(y))>1) else np.nan,
        "auroc":safe_auc(y,p),
        "average_precision":safe_ap(y,p),
        "fpr":fp/(fp+tn) if fp+tn else np.nan,
        "fnr":fn/(fn+tp) if fn+tp else np.nan,
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)
    }

def _finite_or_neginf(x):
    return float(x) if np.isfinite(x) else -np.inf

def select_threshold(y,p):
    rows=[]
    for t in np.linspace(.02,.98,97):
        m=metrics_at_threshold(y,p,t)
        rows.append((
            float(t),
            _finite_or_neginf(m["macro_f1"]),
            _finite_or_neginf(m["mcc"]),
            _finite_or_neginf(m["balanced_accuracy"]),
            -abs(float(t)-0.5),  # deterministic final tie-break
        ))
    return max(rows,key=lambda x:(x[1],x[2],x[3],x[4]))[0]

domains={"HITL-IoT":hitl,"CICIoT2023":cic,"ToN-IoT":ton}

In [ ]:
def run_transfer_matrix(features, analysis_name):
    rows=[]
    fitted={}
    for model_name in ["Logistic Regression","Random Forest"]:
        for source_name,source in domains.items():
            tr=source[source.split=="train"].copy()
            va=source[source.split=="validation"].copy()
            tr=maybe_cap_train(tr,source_name)

            pipe=make_model_pipeline(model_name,features)
            t0=time.perf_counter()
            pipe.fit(tr[features],tr["y"])
            fit_s=time.perf_counter()-t0

            pva=pipe.predict_proba(va[features])[:,1]
            threshold=select_threshold(va["y"],pva)

            fitted[(model_name,source_name)] = (pipe, threshold)

            is_reference_rf = (
                analysis_name == "universal" and model_name == "Random Forest"
            )
            if is_reference_rf or SAVE_ALL_TRANSFER_MODELS:
                model_path = (
                    OUT_DIR / "models" /
                    f"{analysis_name}_{model_name}_{source_name}.joblib".replace(" ","_")
                )
                joblib.dump(pipe, model_path)

                if is_reference_rf:
                    meta = {
                        "analysis": analysis_name,
                        "model": model_name,
                        "source_domain": source_name,
                        "features": list(features),
                        "source_validation_threshold": float(threshold),
                        "source_train_n": int(len(tr)),
                        "model_sha256": sha256_file(model_path),
                    }
                    meta_path = model_path.with_name(model_path.stem + "_metadata.json")
                    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

            for target_name,target in domains.items():
                te=target[target.split=="test"].copy()
                p=pipe.predict_proba(te[features])[:,1]
                met=metrics_at_threshold(te["y"],p,threshold)
                rows.append({
                    "analysis":analysis_name,
                    "eval_subset":"all test",
                    "features":";".join(features),
                    "model":model_name,
                    "train_domain":source_name,
                    "test_domain":target_name,
                    "source_validation_threshold":threshold,
                    "source_train_n":len(tr),
                    "target_test_n":len(te),
                    "fit_seconds":fit_s,
                    **met
                })
                # In-domain result on test rows whose feature vector occurs in neither training nor validation.
                if target_name==source_name and NOVEL_TEST_IDS.get(target_name):
                    nm=te["row_id"].isin(NOVEL_TEST_IDS[target_name]).to_numpy()
                    yn=te["y"].to_numpy()[nm]
                    if nm.sum()>0 and len(np.unique(yn))>1:
                        rows.append({"analysis":analysis_name,"eval_subset":"development-novel test","features":";".join(features),
                                     "model":model_name,"train_domain":source_name,"test_domain":target_name,
                                     "source_validation_threshold":threshold,"source_train_n":len(tr),
                                     "target_test_n":int(nm.sum()),"fit_seconds":fit_s,
                                     **metrics_at_threshold(yn,p[nm],threshold)})
    return pd.DataFrame(rows),fitted

transfer_universal,fitted_universal=run_transfer_matrix(UNIVERSAL_FEATURES,"universal")
transfer_universal.to_csv(OUT_DIR/"tables"/"Table_CrossDomain_Transfer_Universal.csv",index=False)
transfer_universal

In [ ]:
# Pairwise enriched sensitivity analyses.
def run_pairwise(source_a, source_b, features, tag):
    local_domains={source_a:domains[source_a],source_b:domains[source_b]}
    rows=[]
    for model_name in ["Logistic Regression","Random Forest"]:
        for source_name,source in local_domains.items():
            tr=maybe_cap_train(source[source.split=="train"].copy(),source_name)
            va=source[source.split=="validation"].copy()
            pipe=make_model_pipeline(model_name,features)
            pipe.fit(tr[features],tr["y"])
            pva=pipe.predict_proba(va[features])[:,1]
            threshold=select_threshold(va["y"],pva)
            for target_name,target in local_domains.items():
                te=target[target.split=="test"].copy()
                p=pipe.predict_proba(te[features])[:,1]
                rows.append({
                    "analysis":tag,"model":model_name,
                    "train_domain":source_name,"test_domain":target_name,
                    "source_validation_threshold":threshold,
                    **metrics_at_threshold(te["y"],p,threshold)
                })
    return pd.DataFrame(rows)

pair_cic=run_pairwise("HITL-IoT","CICIoT2023",CIC_PAIR_FEATURES,"HITL-CIC enriched")
pair_ton=run_pairwise("HITL-IoT","ToN-IoT",TON_PAIR_FEATURES,"HITL-ToN enriched")
pairwise_transfer=pd.concat([pair_cic,pair_ton],ignore_index=True)
pairwise_transfer.to_csv(OUT_DIR/"tables"/"TableS_CrossDomain_Transfer_Pairwise_Enriched.csv",index=False)
pairwise_transfer

In [ ]:
# ---------------------------------------------------------------------------
# ToN-IoT chronological sensitivity.
#
# This is OPTIONAL and runs only when the supplied network file exposes a
# usable timestamp field. The revision file inspected for this work does not
# contain `ts`, so the expected behavior is to record "not available" rather
# than imply a chronological experiment was completed.
# ---------------------------------------------------------------------------
ton_temporal_rows = []
ton_temporal_status = {
    "requested": bool(RUN_TON_TEMPORAL_SENSITIVITY),
    "usable_timestamp_present": bool(TON_HAS_USABLE_TIMESTAMP),
    "status": None,
    "reason": None,
}

if RUN_TON_TEMPORAL_SENSITIVITY and TON_HAS_USABLE_TIMESTAMP:
    tt_ = ton.sort_values(["timestamp", "row_id"]).reset_index(drop=True)
    n = len(tt_)
    c1, c2 = int(0.70 * n), int(0.85 * n)
    parts = {
        "train": tt_.iloc[:c1],
        "validation": tt_.iloc[c1:c2],
        "test": tt_.iloc[c2:],
    }

    pipe = make_model_pipeline(
        "Random Forest", UNIVERSAL_FEATURES
    ).fit(parts["train"][UNIVERSAL_FEATURES], parts["train"]["y"])

    thr = select_threshold(
        parts["validation"]["y"],
        pipe.predict_proba(parts["validation"][UNIVERSAL_FEATURES])[:, 1],
    )
    p = pipe.predict_proba(parts["test"][UNIVERSAL_FEATURES])[:, 1]

    ton_temporal_rows.append({
        "protocol": "chronological 70/15/15",
        "test_attack_rate": float(parts["test"]["y"].mean()),
        "source_validation_threshold": thr,
        **metrics_at_threshold(parts["test"]["y"], p, thr),
    })

    rnd = transfer_universal[
        (transfer_universal.model == "Random Forest")
        & (transfer_universal.train_domain == "ToN-IoT")
        & (transfer_universal.test_domain == "ToN-IoT")
        & (transfer_universal.eval_subset == "all test")
    ].iloc[0]

    ton_temporal_rows.append({
        "protocol": "stratified random 70/15/15",
        "test_attack_rate": float(ton.loc[ton.split == "test", "y"].mean()),
        "source_validation_threshold": rnd.source_validation_threshold,
        **{
            k: rnd[k] for k in [
                "accuracy", "balanced_accuracy", "precision", "recall",
                "f1", "macro_f1", "mcc", "auroc", "average_precision",
                "fpr", "fnr"
            ]
        },
    })
    ton_temporal_status["status"] = "completed"
    ton_temporal_status["reason"] = "usable timestamp field available"
    print("ToN-IoT chronological vs random split completed.")
else:
    ton_temporal_status["status"] = "not available"
    ton_temporal_status["reason"] = (
        "supplied ToN-IoT network file has no usable timestamp field"
        if not TON_HAS_USABLE_TIMESTAMP
        else "RUN_TON_TEMPORAL_SENSITIVITY=False"
    )
    print("[INFO] ToN chronological sensitivity not available:",
          ton_temporal_status["reason"])

ton_temporal = pd.DataFrame(ton_temporal_rows)
ton_temporal.to_csv(
    OUT_DIR / "tables" / "TableS_ToN_Temporal_Sensitivity.csv",
    index=False
)
(OUT_DIR / "audit" / "ton_temporal_sensitivity_status.json").write_text(
    json.dumps(ton_temporal_status, indent=2), encoding="utf-8"
)

ton_temporal if len(ton_temporal) else pd.DataFrame([ton_temporal_status])

In [ ]:
# Figure 14 — Cross-domain transfer matrices (source-validation thresholds; no target labels used)
fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL, 5.9))
fig.subplots_adjust(hspace=0.55, wspace=0.45)
short = {"HITL-IoT": "HITL-IoT", "CICIoT2023": "CIC", "ToN-IoT": "ToN-IoT"}
allt = transfer_universal[transfer_universal.eval_subset == "all test"]
novel = transfer_universal[transfer_universal.eval_subset == "development-novel test"]
for ax, (model, metric, lbl), letter in zip(axes.flat, [("Random Forest", "macro_f1", "Macro-F1"), ("Random Forest", "mcc", "MCC"),
                                                        ("Logistic Regression", "macro_f1", "Macro-F1"),
                                                        ("Logistic Regression", "mcc", "MCC")], "abcd"):
    tab = (allt[allt.model == model].pivot(index="train_domain", columns="test_domain", values=metric)
           .reindex(index=DOMAIN_ORDER, columns=DOMAIN_ORDER))
    nov = (novel[novel.model == model].set_index("train_domain")[metric] if len(novel) else pd.Series(dtype=float))
    M = tab.to_numpy(float)
    if metric == "mcc":
        im = ax.imshow(np.nan_to_num(M, nan=0.0), cmap=DIV_CMAP, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), aspect="equal")
    else:
        im = ax.imshow(np.nan_to_num(M, nan=0.0), cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="equal")
    for i, src in enumerate(DOMAIN_ORDER):
        for j, tgt in enumerate(DOMAIN_ORDER):
            v = M[i, j]
            dark = (abs(v) > 0.6) if np.isfinite(v) else False
            txt = "undef." if not np.isfinite(v) else f"{v:.2f}"
            ax.text(j, i - (0.12 if i == j and src in nov.index else 0), txt, ha="center", va="center", fontsize=8,
                    fontweight="bold" if i == j else "normal", color="white" if dark else "#1A1A1A")
            if i == j and src in nov.index and np.isfinite(nov[src]):
                ax.text(j, i + 0.22, f"novel {nov[src]:.2f}", ha="center", va="center", fontsize=6.2,
                        color="white" if dark else NEUTRAL)
    ax.set_xticks(range(3), [short[d] for d in DOMAIN_ORDER], fontsize=7.5)
    ax.set_yticks(range(3), [short[d] for d in DOMAIN_ORDER], fontsize=7.5)
    ax.set_xlabel("Test domain", fontsize=8); ax.set_ylabel("Train domain", fontsize=8)
    ax.set_title(f"{model} — {lbl}", fontsize=8.2, pad=4)
    ax.tick_params(length=0)
    for s in ax.spines.values(): s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6.5, length=2)
    add_panel_label(ax, letter, x=-0.42, y=1.02)
CAPTION_FIG14 = (
    "Figure 14. Cross-domain transfer on the harmonised universal feature map. Each cell trains on the row domain and tests on "
    "the column domain; the decision threshold is chosen on the source validation partition only, so no target labels are used. "
    "(a) Random Forest macro-F1, (b) Random Forest Matthews correlation coefficient (MCC; diverging scale centred at 0), "
    "(c) Logistic Regression macro-F1, (d) Logistic Regression MCC. Diagonal cells (bold) are in-domain results; 'novel' reports "
    "the in-domain value on test rows whose harmonised feature vector occurs in neither that domain's training nor validation partition, "
    "guarding against duplicate leakage. 'undef.' marks MCC undefined because a single class was predicted. HITL-IoT diagonal "
    "values use its temporal test partition; the external benchmark is CICIoT2023.")
save_publication_figure(fig, "Figure14_CrossDomain_Transfer_Matrix", CAPTION_FIG14, 4)
plt.show(); plt.close(fig); print(CAPTION_FIG14)


# 5. External calibration and selective prediction

This experiment asks whether the **AI-side mechanism findings from Step 7** generalize beyond HITL-IoT.

For each domain independently:

1. Fit a fixed Random Forest on training data.
2. Split that domain's validation data into calibration-fit and policy-validation subsets.
3. Compare raw, Platt, and isotonic probabilities.
4. Choose the calibration method on policy-validation only.
5. Evaluate calibration and exact workload-controlled selective prediction on the untouched test partition.

The same predeclared deferral budgets are used in all domains. This tests mechanism portability without inventing synthetic human annotations for CICIoT2023 or ToN-IoT.

For comparability with the frozen Step-6 classifier, calibration changes the **review-ranking score** but does not retroactively change the reference 0.50 RF class decision in the selective-prediction tables. Calibration quality itself is reported separately using Brier score, NLL, ECE, AUROC, and average precision.

**Retained-class diagnostic.** Because CICIoT2023 is extremely attack-heavy, aggressive error-focused deferral can leave a retained subset containing only one class. The notebook therefore reports `retained_class_count` and `retained_attack_rate`; MCC remains undefined (`NaN`) for single-class retained subsets even when accuracy is 1.0.

In [ ]:
EPS=1e-7

class PlattCal:
    def fit(self,p,y):
        p=np.clip(np.asarray(p,float),EPS,1-EPS)
        z=np.log(p/(1-p)).reshape(-1,1)
        self.m=LogisticRegression(max_iter=3000,random_state=SEED).fit(z,np.asarray(y,int))
        return self
    def predict(self,p):
        p=np.clip(np.asarray(p,float),EPS,1-EPS)
        z=np.log(p/(1-p)).reshape(-1,1)
        return self.m.predict_proba(z)[:,1]

class IsoCal:
    def fit(self,p,y):
        self.m=IsotonicRegression(y_min=0,y_max=1,out_of_bounds="clip").fit(np.asarray(p,float),np.asarray(y,int))
        return self
    def predict(self,p):
        return np.asarray(self.m.predict(np.asarray(p,float)),float)

def ece_score(y,p,n_bins=ECE_BINS):
    y=np.asarray(y,int); p=np.asarray(p,float)
    bins=np.linspace(0,1,n_bins+1)
    idx=np.clip(np.digitize(p,bins,right=True)-1,0,n_bins-1)
    ece=0.0
    for b in range(n_bins):
        m=idx==b
        if m.any():
            ece += m.mean()*abs(p[m].mean()-y[m].mean())
    return float(ece)

def calibration_metrics(y,p):
    p=np.clip(np.asarray(p,float),EPS,1-EPS); y=np.asarray(y,int)
    return {
        "brier":brier_score_loss(y,p),
        "nll":log_loss(y,np.c_[1-p,p],labels=[0,1]),
        "ece":ece_score(y,p),
        "auroc":safe_auc(y,p),
        "average_precision":safe_ap(y,p)
    }

def exact_topk_mask(score,budget,row_id):
    score=np.asarray(score,float)
    row_id=np.asarray(row_id,str)
    k=int(round(budget*len(score)))
    k=min(max(k,0),len(score))
    order=np.lexsort((row_id,-score))
    mask=np.zeros(len(score),dtype=bool)
    mask[order[:k]]=True
    return mask

def uncertainty(p):
    p=np.asarray(p,float)
    return 1-np.maximum(p,1-p)

def selective_point(y,p,mask):
    y=np.asarray(y,int); p=np.asarray(p,float); mask=np.asarray(mask,bool)
    keep=~mask
    pred=(p>=.5).astype(int)
    err=pred!=y
    met=metrics_at_threshold(y[keep],p[keep],.5) if keep.any() else {}
    y_keep = y[keep]
    return {
        "coverage":float(keep.mean()),
        "deferral_rate":float(mask.mean()),
        "retained_n":int(keep.sum()),
        "retained_class_count":int(np.unique(y_keep).size) if keep.any() else 0,
        "retained_attack_rate":float(y_keep.mean()) if keep.any() else np.nan,
        "error_capture":float((err&mask).sum()/max(err.sum(),1)),
        "selective_risk":1-met.get("accuracy",np.nan),
        "selective_macro_f1":met.get("macro_f1",np.nan),
        "selective_mcc":met.get("mcc",np.nan),
        "selective_fnr":met.get("fnr",np.nan),
    }

def split_validation_for_calibration(df, domain_name):
    # HITL preserves chronology. External datasets use a stratified split because the supplied files lack comparable timestamps.
    if domain_name=="HITL-IoT":
        z=df.sort_values(["timestamp","row_id"]).reset_index(drop=True)
        m=len(z)//2
        return z.iloc[:m].copy(),z.iloc[m:].copy()
    a,b=train_test_split(df,test_size=.50,random_state=SEED,stratify=df["y"])
    return a.copy(),b.copy()

In [ ]:
cal_rows = []
sel_rows = []
within_rf_models = {}
selected_calibration_by_domain = {}

for domain_name, d in domains.items():
    va = d[d.split == "validation"].copy()
    te = d[d.split == "test"].copy()

    # Reuse the already-fitted universal-feature RF from the transfer matrix.
    # This avoids three redundant large RF fits and keeps the calibration
    # experiment tied to the exact source model used in Figure 14.
    pipe, source_threshold = fitted_universal[("Random Forest", domain_name)]
    within_rf_models[domain_name] = pipe

    pva = pipe.predict_proba(va[UNIVERSAL_FEATURES])[:, 1]
    va = va.copy()
    va["p_raw"] = pva
    calfit, policy = split_validation_for_calibration(va, domain_name)

    calibrators = {
        "raw": None,
        "platt": PlattCal().fit(calfit["p_raw"], calfit["y"]),
        "isotonic": IsoCal().fit(calfit["p_raw"], calfit["y"]),
    }

    candidate = []
    for name, cal in calibrators.items():
        pp = (
            policy["p_raw"].to_numpy()
            if cal is None
            else cal.predict(policy["p_raw"])
        )
        candidate.append({
            "method": name,
            **calibration_metrics(policy["y"], pp),
        })

    selected = (
        pd.DataFrame(candidate)
        .sort_values(["brier", "ece", "nll"])
        .iloc[0]["method"]
    )
    selected_calibration_by_domain[domain_name] = selected
    cal = calibrators[selected]

    pte_raw = pipe.predict_proba(te[UNIVERSAL_FEATURES])[:, 1]
    pte_cal = (
        pte_raw
        if cal is None
        else np.clip(cal.predict(pte_raw), 0, 1)
    )

    # Two evaluation subsets:
    #   all test
    #   strict development-novel test (not seen as an exact vector in train/validation)
    eval_subsets = {
        "all test": np.ones(len(te), dtype=bool),
        "development-novel test":
            te["row_id"].isin(NOVEL_TEST_IDS[domain_name]).to_numpy(),
    }

    for eval_subset, smask in eval_subsets.items():
        if smask.sum() == 0:
            continue

        y_sub = te["y"].to_numpy()[smask]
        ids_sub = te["row_id"].to_numpy()[smask]
        raw_sub = pte_raw[smask]
        cal_sub = pte_cal[smask]

        for method, p_sub in [("raw", raw_sub), (selected, cal_sub)]:
            cal_rows.append({
                "domain": domain_name,
                "eval_subset": eval_subset,
                "method": method,
                "n": int(smask.sum()),
                **calibration_metrics(y_sub, p_sub),
            })

        for score_variant, score_name, score_prob in [
            ("raw", "raw uncertainty", raw_sub),
            (
                "calibrated",
                f"{selected} calibrated uncertainty",
                cal_sub,
            ),
        ]:
            score = uncertainty(score_prob)
            for budget in SELECTIVE_BUDGETS:
                mask = exact_topk_mask(score, budget, ids_sub)
                sel_rows.append({
                    "domain": domain_name,
                    "eval_subset": eval_subset,
                    "score_variant": score_variant,
                    "calibration_method": selected,
                    "method": score_name,
                    "budget": budget,
                    "n": int(smask.sum()),
                    **selective_point(y_sub, raw_sub, mask),
                })

calibration_external = pd.DataFrame(cal_rows)
selective_external = pd.DataFrame(sel_rows)

calibration_external.to_csv(
    OUT_DIR / "tables" / "Table_External_Calibration.csv", index=False
)
selective_external.to_csv(
    OUT_DIR / "tables" / "Table_External_Selective_Prediction.csv", index=False
)

print("Calibration results:")
display(calibration_external)

print("\nSelective prediction at 8% review:")
display(
    selective_external[
        np.isclose(selective_external.budget, .08)
    ].sort_values(["domain", "eval_subset", "score_variant"])
)

In [ ]:
# Continuous AURC from exact score ordering.
def aurc(y, p, score, row_id):
    y = np.asarray(y, int)
    p = np.asarray(p, float)
    score = np.asarray(score, float)
    row_id = np.asarray(row_id, str)

    order = np.lexsort((row_id, -score))
    pred = (p >= .5).astype(int)
    errors = (pred != y).astype(float)

    coverages, risks = [], []
    deferred = np.zeros(len(y), dtype=bool)

    for frac in np.linspace(0, 1, 201):
        k = int(round(frac * len(y)))
        deferred[:] = False
        deferred[order[:k]] = True
        keep = ~deferred
        if keep.sum() == 0:
            continue
        coverages.append(keep.mean())
        risks.append(errors[keep].mean())

    ix = np.argsort(coverages)
    return float(
        np.trapezoid(
            np.asarray(risks)[ix],
            np.asarray(coverages)[ix]
        )
    )

aurc_rows = []

for domain_name, d in domains.items():
    va = d[d.split == "validation"].copy()
    te = d[d.split == "test"].copy()
    pipe = within_rf_models[domain_name]

    pva = pipe.predict_proba(va[UNIVERSAL_FEATURES])[:, 1]
    va2 = va.copy()
    va2["p_raw"] = pva
    calfit, policy = split_validation_for_calibration(va2, domain_name)

    cals = {
        "raw": None,
        "platt": PlattCal().fit(calfit["p_raw"], calfit["y"]),
        "isotonic": IsoCal().fit(calfit["p_raw"], calfit["y"]),
    }

    candidates = []
    for name, cal in cals.items():
        pp = (
            policy["p_raw"].to_numpy()
            if cal is None
            else cal.predict(policy["p_raw"])
        )
        candidates.append({
            "method": name,
            **calibration_metrics(policy["y"], pp),
        })
    selected = (
        pd.DataFrame(candidates)
        .sort_values(["brier", "ece", "nll"])
        .iloc[0]["method"]
    )

    p_raw = pipe.predict_proba(te[UNIVERSAL_FEATURES])[:, 1]
    p_cal = (
        p_raw
        if cals[selected] is None
        else np.clip(cals[selected].predict(p_raw), 0, 1)
    )

    eval_subsets = {
        "all test": np.ones(len(te), dtype=bool),
        "development-novel test":
            te["row_id"].isin(NOVEL_TEST_IDS[domain_name]).to_numpy(),
    }

    for eval_subset, smask in eval_subsets.items():
        if smask.sum() == 0:
            continue

        yy = te["y"].to_numpy()[smask]
        ids = te["row_id"].to_numpy()[smask]
        raw_sub = p_raw[smask]
        cal_sub = p_cal[smask]

        aurc_rows.append({
            "domain": domain_name,
            "eval_subset": eval_subset,
            "method": "raw RF uncertainty",
            "n": int(smask.sum()),
            "aurc": aurc(
                yy, raw_sub, uncertainty(raw_sub), ids
            ),
        })
        aurc_rows.append({
            "domain": domain_name,
            "eval_subset": eval_subset,
            "method": f"{selected} calibrated uncertainty",
            "n": int(smask.sum()),
            "aurc": aurc(
                yy, raw_sub, uncertainty(cal_sub), ids
            ),
        })

aurc_table = pd.DataFrame(aurc_rows)
aurc_table.to_csv(
    OUT_DIR / "tables" / "Table_External_AURC.csv", index=False
)
aurc_table

In [ ]:
# Figure 15 — Calibration and selective prediction on each domain.
# Panels (a–c): headline all-test risk–coverage curves.
# Panel (d): validation-selected calibrated uncertainty at 8% review,
# comparing all-test and strict development-novel test subsets.
fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL, 5.7))
fig.subplots_adjust(hspace=0.50, wspace=0.34)

for ax, dom, letter in zip(
    [axes[0, 0], axes[0, 1], axes[1, 0]],
    DOMAIN_ORDER,
    "abc"
):
    g = selective_external[
        (selective_external.domain == dom)
        & (selective_external.eval_subset == "all test")
    ]

    for score_variant, m in g.groupby("score_variant"):
        m = m.sort_values("coverage")
        calibrated = score_variant == "calibrated"
        label = (
            f"{m.iloc[0]['calibration_method']} calibrated"
            if calibrated
            else "raw"
        )
        ax.plot(
            100 * m["coverage"],
            100 * m["selective_risk"],
            marker="s" if calibrated else "o",
            ms=3.8,
            lw=1.3,
            ls="-" if calibrated else "--",
            color=DOMAIN_COLORS[dom],
            alpha=1.0 if calibrated else 0.70,
            label=label,
        )

    ax.set_xlabel("Coverage (% retained)")
    ax.set_ylabel("Selective risk (%)")
    ax.invert_xaxis()
    ax.set_ylim(bottom=0)
    if dom == "ToN-IoT":
        ax.text(0.04, 0.94, dom, transform=ax.transAxes, ha="left", va="top", fontsize=8, color=DOMAIN_COLORS[dom], fontweight="bold")
        ax.legend(loc="upper right", fontsize=6.8)
    else:
        ax.text(0.97, 0.94, dom, transform=ax.transAxes, ha="right", va="top", fontsize=8, color=DOMAIN_COLORS[dom], fontweight="bold")
        ax.legend(loc="lower left", fontsize=6.8)
    ygrid(ax, "both")
    add_panel_label(ax, letter)

# Panel (d): duplicate-robust 8% error-capture comparison.
ax = axes[1, 1]
eight = selective_external[
    np.isclose(selective_external.budget, 0.08)
    & (selective_external.score_variant == "calibrated")
].copy()

yy = np.arange(3)[::-1]
for y_, dom in zip(yy, DOMAIN_ORDER):
    gd = eight[eight.domain == dom]
    for subset, marker, face, offset in [
        ("all test", "o", DOMAIN_COLORS[dom], +0.12),
        ("development-novel test", "s", "white", -0.12),
    ]:
        r = gd[gd.eval_subset == subset]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        v = 100 * r["error_capture"]
        ax.plot(
            v, y_ + offset,
            marker=marker,
            ms=6.2,
            mfc=face,
            mec=DOMAIN_COLORS[dom],
            mew=1.0,
            ls="",
            zorder=3,
        )
        ax.text(
            v + 2.5 if v < 85 else v - 2.5,
            y_ + offset,
            f"{v:.1f}%",
            va="center",
            ha="left" if v < 85 else "right",
            fontsize=6.5,
            color=NEUTRAL,
        )

ax.axvline(8, ls=":", lw=0.9, color="#9A9A9A")
ax.text(9.0, 2.36, "random (8%)", fontsize=6.5, color=NEUTRAL, va="center")
ax.set_yticks(yy, DOMAIN_ORDER, fontsize=7.5)
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, 104)
ax.set_ylim(-0.6, 2.5)
ax.set_xlabel("RF errors captured at 8% review (%)")
ax.legend(
    handles=[
        Line2D([], [], marker="o", ls="", mfc=NEUTRAL, mec=NEUTRAL,
               label="all test"),
        Line2D([], [], marker="s", ls="", mfc="white", mec=NEUTRAL,
               label="development-novel test"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.20),
    ncol=2,
    fontsize=6.8,
)
ygrid(ax, "x")
add_panel_label(ax, "d", x=-0.28)

CAPTION_FIG15 = 'Figure 15. External selective-prediction robustness using the universal-feature within-domain Random Forest. (a–c) Risk–coverage curves on each full test partition under exact workload control. (d) Fraction of RF errors captured at a fixed 8% review budget on full-test and development-novel subsets. Calibration is fitted and selected on validation data only. CICIoT2023 is used as the external IoT benchmark.'

save_publication_figure(
    fig,
    "Figure15_External_Selective_Prediction",
    CAPTION_FIG15,
    4
)
plt.show()
plt.close(fig)
print(CAPTION_FIG15)

In [ ]:
# Figure quality control (machine-checkable properties only)
REQUIRED_FIGS = {"Figure13_External_Domain_Characterization": 4, "Figure14_CrossDomain_Transfer_Matrix": 4,
                 "Figure15_External_Selective_Prediction": 4}
BANNED = [r"ml_confidence", r"human confidence", r"real analyst", r"state[- ]of[- ]the[- ]art", r"proves",
          r"validates the realism", r"realistic"]
qc_rows = []
for stem, n_pan in REQUIRED_FIGS.items():
    info = FIGURE_REGISTRY[stem]; p = info["paths"]; img = plt.imread(p["png"])
    text = (info["caption"] + " " + info["texts"]).lower()
    row = {"figure": stem, "pdf": p["pdf"].stat().st_size > 5000 and p["pdf"].read_bytes()[:4] == b"%PDF",
           "svg": p["svg"].stat().st_size > 5000, "png_dpi_ok": img.shape[1] / info["size_in"][0] >= 290,
           "panels_expected": info["n_panels"] == n_pan,
           "axes_labelled": all((a["xlabel"] or a["ylabel"]) or a["named_ticks"] for a in info["axes"]),
           "terminology_ok": not any(re.search(b, text) for b in BANNED), "png_px": f"{img.shape[1]}×{img.shape[0]}"}
    row["result"] = "PASS" if all(v for k, v in row.items() if k not in ("figure", "png_px")) else "FAIL"
    qc_rows.append(row)
figure_qc = pd.DataFrame(qc_rows); figure_qc.to_csv(OUT_DIR / "audit" / "Step8_figure_quality_control.csv", index=False)
FIGURES_OK = bool((figure_qc.result == "PASS").all())
print("FIGURE QUALITY CONTROL\n")
for r in qc_rows:
    print(f"{r['figure'] + ':':48s}{r['result']}  ({r['png_px']})")
print("\n(Aesthetic quality is not machine-verifiable — manual checklist:)")
for item in ["Text readable at final journal width.", "Domain colours consistent across Figures 13–15.",
             "Interpretable in greyscale.", "Legends do not obscure data.", "Captions match the analyses."]:
    print(f"[ ] {item}")
figure_qc


# 6. Cross-domain uncertainty sensitivity

Within-domain selective prediction establishes whether the Step-7 mechanism family is useful on independent datasets.

The following supplementary analysis asks a harder question:

> When a classifier is trained in one domain and evaluated in a different domain, does its own uncertainty still concentrate its target-domain errors?

No target-label threshold tuning is performed. This is an uncertainty-transfer stress test under domain shift, not a deployment claim.

In [ ]:
cross_uncertainty_rows=[]
for (model_name,source_name),(pipe,threshold) in fitted_universal.items():
    if model_name!="Random Forest":
        continue
    for target_name,target in domains.items():
        if target_name==source_name:
            continue
        te=target[target.split=="test"].copy()
        p=pipe.predict_proba(te[UNIVERSAL_FEATURES])[:,1]
        pred=(p>=threshold).astype(int)
        score=1-np.maximum(p,1-p)
        err=(pred!=te["y"].to_numpy()).astype(int)
        for budget in [.05,.08,.10,.20]:
            mask=exact_topk_mask(score,budget,te["row_id"])
            cross_uncertainty_rows.append({
                "train_domain":source_name,"test_domain":target_name,
                "budget":budget,"realized_deferral":mask.mean(),
                "target_error_rate":err.mean(),
                "error_capture":float((err&mask).sum()/max(err.sum(),1)),
            })

cross_uncertainty=pd.DataFrame(cross_uncertainty_rows)
cross_uncertainty.to_csv(OUT_DIR/"tables"/"TableS_CrossDomain_Uncertainty_Transfer.csv",index=False)
cross_uncertainty

# 7. Reviewer-facing interpretation

A credible external-validation result may show **both correspondence and divergence**.

Examples of defensible conclusions:

- If class-conditional feature distances are small for some variables and large for others, report a **mixed correspondence profile**, not "realistic" or "unrealistic" as a blanket label.
- If within-domain performance is high but cross-domain transfer falls substantially, interpret that as a **domain gap** and evidence that HITL-IoT should complement rather than replace capture-based datasets.
- If uncertainty-based selective prediction still concentrates errors on CICIoT2023 and ToN-IoT, that supports the **portability of the mechanism family** independently of generated oversight variables.
- If uncertainty transfer degrades under cross-domain training/testing, report it as a limitation of policy transfer and a motivation for dataset-specific calibration.

External traffic validation does **not** validate empirical human behaviour. That limitation remains explicit.

In [ ]:
# Generate concise manuscript/reviewer evidence from completed outputs.
rf_transfer = transfer_universal[
    (transfer_universal.model == "Random Forest")
    & (transfer_universal.eval_subset == "all test")
].copy()

nv_transfer = transfer_universal[
    (transfer_universal.model == "Random Forest")
    & (transfer_universal.eval_subset == "development-novel test")
].copy()

summary_lines = [
    "# Step 8 — External validation evidence",
    "",
    "## Data sources",
    f"- CICIoT2023: attack prevalence {cic.y.mean():.3%}; exact local file identity and SHA-256 are recorded in the audit.",
    f"- ToN-IoT Network: {len(ton):,} records; attack prevalence {ton.y.mean():.3%}.",
    f"- HITL-IoT v1.1: {len(hitl):,} records; attack prevalence {hitl.y.mean():.3%}.",
    "",
    "## Protocol",
    "- HITL-IoT uses the frozen temporal train/validation/test partition.",
    "- External datasets use fixed-seed stratified 70/15/15 internal partitions.",
    "- Cross-domain thresholds are selected on source validation only.",
    "- Primary transfer uses the same conservative universal feature map in all three domains.",
    "- Duplicate-sensitive in-domain results are repeated on development-novel test rows whose exact harmonized feature vector appears in neither training nor validation.",
    "",
    "## Random-Forest transfer summary",
]

for _, r in rf_transfer.iterrows():
    mcc_text = "undefined" if pd.isna(r.mcc) else f"{r.mcc:.4f}"
    summary_lines.append(
        f"- {r.train_domain} → {r.test_domain}: "
        f"Macro-F1={r.macro_f1:.4f}, MCC={mcc_text}, AUROC={r.auroc:.4f}."
    )

summary_lines += ["", "## Duplicate-leakage robustness"]
for _, r in duplicate_audit.iterrows():
    summary_lines.append(
        f"- {r.domain}: {r.rows_in_duplicated_vectors_pct:.1f}% of rows share a "
        f"harmonized feature vector; {r.test_vectors_also_in_train_pct:.1f}% of "
        f"test vectors also occur in training and "
        f"{r.test_vectors_also_in_development_pct:.1f}% occur in training/validation."
    )

for _, r in nv_transfer.iterrows():
    mcc_text = "undefined" if pd.isna(r.mcc) else f"{r.mcc:.4f}"
    summary_lines.append(
        f"- {r.train_domain} in-domain RF on development-novel test rows "
        f"(n={int(r.target_test_n):,}): Macro-F1={r.macro_f1:.4f}, MCC={mcc_text}."
    )

summary_lines += ["", "## Domain separability"]
for _, r in domain_auc.iterrows():
    summary_lines.append(
        f"- {r.pair}: exact-vector-deduplicated domain-discriminator AUROC "
        f"{r.domain_classifier_auroc:.3f} (logistic) and "
        f"{r.domain_classifier_auroc_rf:.3f} (random forest)."
    )

summary_lines.append(
    "- HITL-IoT ttl is a per-device constant by design and is excluded from every model."
)

if len(ton_temporal):
    for _, r in ton_temporal.iterrows():
        mcc_text = "undefined" if pd.isna(r.mcc) else f"{r.mcc:.4f}"
        summary_lines.append(
            f"- ToN-IoT in-domain RF, {r.protocol}: "
            f"Macro-F1={r.macro_f1:.4f}, MCC={mcc_text}."
        )
else:
    summary_lines.append(
        "- ToN-IoT chronological sensitivity was not run because the supplied "
        "network file contains no usable timestamp field."
    )

summary_lines += ["", "## Selective-prediction duplicate robustness"]
eight = selective_external[
    np.isclose(selective_external.budget, .08)
    & (selective_external.score_variant == "calibrated")
]
for dom in DOMAIN_ORDER:
    g = eight[eight.domain == dom]
    vals = {}
    for subset in ["all test", "development-novel test"]:
        r = g[g.eval_subset == subset]
        if len(r):
            vals[subset] = 100 * float(r.iloc[0].error_capture)
    if vals:
        summary_lines.append(
            f"- {dom}: validation-selected calibrated uncertainty captured "
            + ", ".join(f"{v:.1f}% of RF errors on {k}" for k, v in vals.items())
            + " at an exact 8% review budget."
        )

summary_lines += [
    "",
    "## Interpretation boundary",
    "These experiments quantify distributional correspondence, domain shift, "
    "transfer utility, calibration, and selective-prediction portability. "
    "They do not establish empirical human-behaviour validity, operational SOC "
    "performance, or one-to-one equivalence between synthetic and capture-derived features."
]

evidence_text = "\n".join(summary_lines)
(OUT_DIR / "manuscript_values" / "step8_external_validation_evidence.md").write_text(
    evidence_text, encoding="utf-8"
)
print(evidence_text)

In [ ]:
# Reproducibility manifest and final audit.
env = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform(),
    "seed": SEED,
    "fast_mode": FAST_MODE,
    "rf_trees": RF_TREES,
    "rf_min_samples_leaf": RF_MIN_SAMPLES_LEAF,
    "cic_description": "CICIoT2023",
    "cic_specific_redistribution_inferred": False,
    "ton_usable_timestamp_present": TON_HAS_USABLE_TIMESTAMP,
    "hitl_dataset_sha256":
        sha256_file(HITL_RELEASE / "HITL-IoT_dataset_v1.1.csv"),
    "cic_file_sha256": sha256_file(CIC_FILE),
    "ton_network_sha256": sha256_file(TON_NETWORK_FILE),
    "step6_context_fingerprint_observed": STEP6_SELECTED_REPORT.get("context_fingerprint"),
    "step6_legacy_freeze_fingerprint": LEGACY_EXPECTED_STEP6_FINGERPRINT,
    "step6_invariants_verified": bool(STEP6_SELECTED_REPORT.get("valid")),
}
(OUT_DIR / "logs" / "environment.json").write_text(
    json.dumps(env, indent=2), encoding="utf-8"
)

rf_reference_models = [
    OUT_DIR / "models" / f"universal_Random_Forest_{dom}.joblib".replace(" ", "_")
    for dom in DOMAIN_ORDER
]

checks = {
    "Canonical HITL hash verified":
        env["hitl_dataset_sha256"] == EXPECTED_HITL_SHA256,
    "Frozen Step-6 scientific invariants verified":
        STEP6_DIR is not None and bool(STEP6_SELECTED_REPORT.get("valid")),
    "CICIoT2023 supplied derivative loaded":
        len(cic) > 0,
    "ToN-IoT Network loaded":
        len(ton) > 0,
    "Universal feature map complete":
        all(f in hitl and f in cic and f in ton for f in UNIVERSAL_FEATURES),
    "TTL excluded from every model feature set":
        "ttl" not in CIC_PAIR_FEATURES
        and "ttl" not in UNIVERSAL_FEATURES
        and "ttl" not in TON_PAIR_FEATURES,
    "Destination port represented as categorical bands":
        all(f in TON_PAIR_FEATURES for f in TON_PORT_BAND_FEATURES)
        and "dst_port" not in TON_PAIR_FEATURES,
    "Class-conditional distribution metrics generated":
        len(dist_metrics) > 0,
    "Domain discriminator generated":
        len(domain_auc) == 2,
    "Domain discriminator deduplicated before split":
        bool(domain_auc["deduplicated_before_split"].all()),
    "3x3 transfer matrix generated":
        int((transfer_universal.eval_subset == "all test").sum()) == 18,
    "Development-novel in-domain transfer results generated":
        int((transfer_universal.eval_subset == "development-novel test").sum()) > 0,
    "External provenance recorded conservatively":
        len(provenance) == 2
        and not bool(provenance.loc[
            provenance.dataset == "CICIoT2023",
            "specific_redistribution_inferred_from_row_count"
        ].iloc[0]),
    "Duplicate-leakage audit generated":
        len(duplicate_audit) == 3,
    "Strict development-novel test IDs available":
        all(len(NOVEL_TEST_IDS[d]) > 0 for d in DOMAIN_ORDER),
    "Three within-domain RF reference models serialized":
        all(p.exists() for p in rf_reference_models),
    "External calibration generated":
        len(calibration_external) > 0,
    "Development-novel calibration generated":
        int((calibration_external.eval_subset == "development-novel test").sum()) > 0,
    "Selective prediction generated":
        len(selective_external) > 0,
    "Development-novel selective prediction generated":
        int((selective_external.eval_subset == "development-novel test").sum()) > 0,
    "AURC generated for full and development-novel subsets":
        set(aurc_table["eval_subset"]) >= {"all test", "development-novel test"},
    "Cross-domain uncertainty sensitivity generated":
        len(cross_uncertainty) > 0,
    "ToN temporal analysis status recorded":
        (OUT_DIR / "audit" / "ton_temporal_sensitivity_status.json").exists(),
    "Publication figures 13–15 QC-checked":
        FIGURES_OK,
}

audit = pd.DataFrame([
    {"check": k, "status": "PASS" if v else "FAIL"}
    for k, v in checks.items()
])
audit.to_csv(OUT_DIR / "audit" / "final_audit.csv", index=False)

# Manifest excludes itself to avoid self-hash inconsistency.
files = [
    p for p in OUT_DIR.rglob("*")
    if p.is_file() and p.name != "step8_manifest_sha256.csv"
]
manifest = pd.DataFrame([
    {
        "path": str(p.relative_to(OUT_DIR)),
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    }
    for p in sorted(files)
])
manifest.to_csv(
    OUT_DIR / "step8_manifest_sha256.csv", index=False
)

print("=" * 78)
print("HITL-IoT v1.1 — STEP 8 EXTERNAL VALIDATION")
print("=" * 78)
for k, v in checks.items():
    print(f"{k + ':':62s} {'PASS' if v else 'FAIL'}")

print(f"\nAudit checks: {sum(checks.values())}/{len(checks)} PASS")
assert all(checks.values()), [k for k, v in checks.items() if not v]
print("\n[PASS] Step 8 complete. Freeze outputs before manuscript integration.")

## Main paper output recommendation

For the revised manuscript, prioritize:

- **Figure 13:** external-domain characterization and class-conditional distributional correspondence, including duplicate-robust linear/nonlinear domain discrimination.
- **Figure 14:** 3×3 cross-domain transfer matrix, with strict development-novel diagonal sensitivity values.
- **Figure 15:** calibration/selective-prediction portability and 8% error-capture robustness on both full-test and development-novel external records.

Recommended primary tables:

- External source/provenance and semantic feature-mapping table.
- Universal-feature Random-Forest transfer table, including development-novel diagonal sensitivity.
- Calibration/selective-prediction summary at the predeclared 8% review budget for both full-test and development-novel subsets.

Recommended supplementary material:

- Logistic Regression transfer results.
- HITL↔CIC and HITL↔ToN enriched-feature sensitivity analyses.
- Detailed KS/JSD/Wasserstein and correlation-distance tables.
- AURC for full and development-novel test subsets.
- Cross-domain uncertainty-transfer stress test.
- ToN temporal sensitivity only if a usable timestamp field is actually present.

Do not describe the external analysis as proving that HITL-IoT is "realistic." Report correspondence and domain gaps directly.